In [ ]:
import os

os.listdir('/content')

In [ ]:
import json

notebook_path = "/content/your_notebook.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb.get("metadata", {}).pop("widgets", None)

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
# تحديد مسار الملف
file_path = '/content/drive/MyDrive/IT Support Ticket Data.csv'

# قراءة الملف (index_col=0 لإزالة عمود Unnamed: 0)
df = pd.read_csv(file_path, index_col=0)

# عرض البيانات
display(df.head())

## Data Structure Overview

### Subtask:
Examine the dataframe's shape, column types, and basic statistics to understand the overall dataset scale.


**Reasoning**:
I will provide the code to inspect the DataFrame structure, including its info, shape, and detailed descriptive statistics for all column types.



In [ ]:
print('--- DataFrame Info ---')
df.info()

print('\n--- DataFrame Shape ---')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

print('\n--- Descriptive Statistics ---')
display(df.describe(include='all'))

In [ ]:
print('--- Missing Values Count ---')
missing_values = df.isnull().sum()
print(missing_values)

print('\n--- Duplicated Rows Count ---')
duplicate_count = df.duplicated().sum()
print(f'Total duplicate rows: {duplicate_count}')

In [ ]:
# إزالة الصفوف المكررة
df = df.drop_duplicates().reset_index(drop=True)
print(f'Shape after removing duplicates: {df.shape}')
print(f'Rows removed: {29651 - len(df)}')

In [ ]:
# إزالة الصف الناقص في عمود Body
df = df.dropna(subset=['Body']).reset_index(drop=True)
print(f'Shape after removing missing values: {df.shape}')
print(f'Rows removed: 1')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from collections import Counter
import pandas as pd

# 1. Calculate counts and percentages for Department and Priority
dept_counts = df['Department'].value_counts()
dept_percent = df['Department'].value_counts(normalize=True) * 100
prio_counts = df['Priority'].value_counts()
prio_percent = df['Priority'].value_counts(normalize=True) * 100

print('--- Department Distribution ---')
display(pd.DataFrame({'Count': dept_counts, 'Percentage (%)': dept_percent}))

print('\n--- Priority Distribution ---')
display(pd.DataFrame({'Count': prio_counts, 'Percentage (%)': prio_percent}))

# 2. Visualization of distributions (Department and Priority)
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.countplot(data=df, y='Department', ax=axes[0], hue='Department', palette='viridis', order=dept_counts.index, legend=False)
axes[0].set_title('Distribution of Support Tickets by Department')
axes[0].set_xlabel('Number of Tickets')

sns.countplot(data=df, x='Priority', ax=axes[1], hue='Priority', palette='magma', order=prio_counts.index, legend=False)
axes[1].set_title('Distribution of Support Tickets by Priority')
axes[1].set_ylabel('Number of Tickets')

plt.tight_layout()
plt.show()

# 3. Parse 'Tags' column and Tag statistics
def parse_tags(tag_str):
    try:
        return ast.literal_eval(tag_str)
    except:
        return []

df['Tags_cleaned'] = df['Tags'].apply(parse_tags)
num_tags_per_ticket = df['Tags_cleaned'].apply(len)

flattened_tags = [tag for sublist in df['Tags_cleaned'] for tag in sublist]
tag_freq = Counter(flattened_tags)
unique_tags_count = len(tag_freq)
common_tags = tag_freq.most_common(20)

print(f'\nTotal Unique Tags: {unique_tags_count}')
print(f'Average tags per ticket: {num_tags_per_ticket.mean():.2f}')

# 4. Visualization of Top 20 Tags
tag_freq_df = pd.DataFrame(common_tags, columns=['Tag', 'Frequency'])

plt.figure(figsize=(12, 8))
sns.barplot(data=tag_freq_df, x='Frequency', y='Tag', hue='Tag', palette='coolwarm', legend=False)
plt.title('Top 20 Most Frequent Tags')
plt.xlabel('Frequency (Number of Occurrences)')
plt.ylabel('Tag')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Statistical Breakdown
dept_counts = df['Department'].value_counts()
dept_percent = df['Department'].value_counts(normalize=True) * 100
dept_stats = pd.DataFrame({'Count': dept_counts, 'Percentage (%)': dept_percent.round(2)})

print('--- Professional Department Distribution Analysis ---')
display(dept_stats)

# 2. Enhanced Multi-Graph Visualization
sns.set_theme(style='whitegrid', palette='muted')
fig, ax = plt.subplots(1, 2, figsize=(20, 8))

# A. Bar Chart with Annotations
sns.barplot(x=dept_stats['Count'], y=dept_stats.index, ax=ax[0], palette='viridis', hue=dept_stats.index, legend=False)
ax[0].set_title('Volume of Tickets by Department', fontsize=16, fontweight='bold')
ax[0].set_xlabel('Total Ticket Count', fontsize=12)
ax[0].set_ylabel('Department Name', fontsize=12)

# Add percentage labels to bars
for i, v in enumerate(dept_stats['Count']):
    ax[0].text(v + 50, i, f"{dept_stats['Percentage (%)'].iloc[i]}%", color='black', va='center', fontweight='bold')

# B. Pie Chart for Proportional Share
ax[1].pie(dept_stats['Count'], labels=dept_stats.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'), explode=[0.05 if i == 0 else 0 for i in range(len(dept_stats))])
ax[1].set_title('Proportional Distribution of Tickets', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

# 3. Insightful Summary
print('\n' + '='*50)
print('KEY EDA INSIGHTS')
print('='*50)
print(f'• Market Leader: {dept_stats.index[0]} dominates with {dept_stats["Percentage (%)"].iloc[0]}% of total volume.')
print(f'• Tail End: {dept_stats.index[-1]} represents the smallest segment at {dept_stats["Percentage (%)"].iloc[-1]}%.')
print(f'• Skewness: The top 3 departments handle over 60% of the total workload.')
print('='*50)

## Class Imbalance Analysis

Quantifying the imbalance ratio is critical for choosing the right modeling strategy.

In [ ]:
# === Class Imbalance Ratio ===
dept_counts = df['Department'].value_counts()
largest_class = dept_counts.iloc[0]
smallest_class = dept_counts.iloc[-1]

imbalance_ratio = largest_class / smallest_class

print(f"Largest class:  {dept_counts.index[0]} ({largest_class} samples)")
print(f"Smallest class: {dept_counts.index[-1]} ({smallest_class} samples)")
print(f"Imbalance Ratio: {imbalance_ratio:.1f}x")
print()
print("--- Imbalance Summary ---")
for dept, count in dept_counts.items():
    ratio_to_largest = largest_class / count
    print(f"  {dept:40s} {count:5d}  ({ratio_to_largest:.1f}x smaller than largest)")

print(f"\n→ The dataset has a {imbalance_ratio:.0f}:1 imbalance ratio.")
print("→ This means class_weight='balanced' or stratified sampling is essential.")

In [ ]:
df['wc'] = df['Body'].apply(lambda x: len(str(x).split()))
print(df['wc'].describe())
print(f"< 10 words: {(df['wc']<10).sum()} ({(df['wc']<10).sum()/len(df)*100:.1f}%)")
print(f"> 200 words: {(df['wc']>200).sum()} ({(df['wc']>200).sum()/len(df)*100:.1f}%)")

In [ ]:
import re

# 1. Sample 10 random entries for manual inspection
sample_bodies = df['Body'].dropna().sample(10, random_state=42)
print('--- Sample Ticket Bodies ---')
for i, text in enumerate(sample_bodies, 1):
    print(f'Sample {i}:\n{text[:300]}...')
    print('-' * 40)

# 2. Define regex patterns
html_pattern = r'<[^>]+>'
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'

# 3. Apply patterns to count occurrences
# We fillna with empty string to avoid errors during regex search
body_series = df['Body'].fillna('')

has_html = body_series.str.contains(html_pattern, regex=True).sum()
has_url = body_series.str.contains(url_pattern, regex=True).sum()

print('\n--- Text Cleaning Requirements Analysis ---')
print(f'Total rows with HTML tags: {has_html}')
print(f'Total rows with URLs: {has_url}')

# 4. Check for common boilerplate examples
boilerplate_examples = [
    'Dear Customer Support Team',
    'Kind regards',
    'I hope this message finds you well',
    'Thank you for your assistance'
]

print('\n--- Boilerplate Detection Check ---')
for phrase in boilerplate_examples:
    count = body_series.str.contains(phrase, case=False).sum()
    print(f'Rows containing "{phrase}": {count}')

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import re
from collections import Counter

# 1. Prepare data (using the existing df from the pipeline)
# Ensure we only analyze rows with valid 'Body' content
analysis_bodies = df['Body'].dropna().astype(str)

print("="*40)
print("1. Automated Frequent Phrase Detection (N-Grams)")
print("="*40)

# Use CountVectorizer to find sequences of 4 to 6 words
vec = CountVectorizer(ngram_range=(4, 6), stop_words=None, max_features=20)
X = vec.fit_transform(analysis_bodies)
counts = X.sum(axis=0).A1
vocab = vec.get_feature_names_out()

# Display results in a sorted table
freq_df = pd.DataFrame({'N-Gram Phrase': vocab, 'Frequency': counts})
freq_df = freq_df.sort_values(by='Frequency', ascending=False)
print(freq_df.head(10).to_string(index=False))

print("\n" + "="*40)
print("2. Automated Exact Sentence Detection")
print("="*40)

# Function to split text into sentences based on punctuation
def get_sentences(text):
    parts = re.split(r'[.!?\n]+', text)
    # Keep sentences with 3 or more words to avoid single words/noise
    return [p.strip() for p in parts if len(p.strip().split()) >= 3]

all_sentences = []
for text in analysis_bodies:
    all_sentences.extend(get_sentences(text))

# Count the most frequent sentences
sentence_counts = Counter(all_sentences)

# Print top 10 most frequent sentences across the data
for phrase, count in sentence_counts.most_common(10):
    print(f'Count: {count} | Phrase: "{phrase}"')

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(stop_words='english')
X = vec.fit_transform(df['Body'].dropna())

print("Vocabulary size:", len(vec.vocabulary_))

## Text Length Analysis



In [ ]:
df['wc'] = df['Body'].apply(lambda x: len(str(x).split()))
print(df['wc'].describe())
print(f"< 10 words: {(df['wc']<10).sum()} ({(df['wc']<10).sum()/len(df)*100:.1f}%)")
print(f"> 200 words: {(df['wc']>200).sum()} ({(df['wc']>200).sum()/len(df)*100:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate word counts handling the missing value
df['word_count'] = df['Body'].fillna('').apply(lambda x: len(str(x).split()))

# 2. Retrieve descriptive statistics
word_count_stats = df['word_count'].describe()
print('--- Word Count Statistics ---')
print(word_count_stats)

# 3. Visualize the distribution
plt.figure(figsize=(12, 6))

# Main Histogram
sns.histplot(df['word_count'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of Ticket Body Word Counts')
plt.xlabel('Word Count')
plt.ylabel('Frequency')

# Add mean and median lines for context
plt.axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.2f}')
plt.axvline(df['word_count'].median(), color='green', linestyle='-', label=f'Median: {df["word_count"].median():.2f}')
plt.legend()

plt.show()

# 4. Boxplot to highlight outliers
plt.figure(figsize=(12, 4))
sns.boxplot(x=df['word_count'], palette='pastel')
plt.title('Boxplot of Word Counts (Identifying Outliers)')
plt.xlabel('Word Count')
plt.show()

In [ ]:
# حساب عدد الكلمات في عمود Body
df['word_count'] = df['Body'].apply(lambda x: len(str(x).split()))

# استخراج الصفوف اللي فيها أقل من 10 كلمات
short_rows = df[df['word_count'] < 4].copy()

# شكل الداتا دي
print("Shape of rows with less than 4 words:", short_rows.shape)

# عرض الأعمدة المهمة
display(short_rows[['Body', 'word_count','Department']].head(20))

ملاحظه ::مش خمسحها لكن هشوف تأثيرها ع التوقع ف الموديل ف الماتريكس

واعمل ماتريكس للتكتات الكبيره و ماتريكس للتيكتات الصغيره عشان اشوف الموديل بيقدر يصنف ايه احسن

افسمهم ع اساس ال median

In [ ]:
pd.set_option('display.max_colwidth', None)

display(short_rows[['Body', 'word_count','Department']])

In [ ]:
# حساب عدد الكلمات في عمود Body
df['word_count'] = df['Body'].apply(lambda x: len(str(x).split()))

# استخراج الصفوف اللي فيها أقل من 10 كلمات
short_rows = df[df['word_count'] < 3].copy()

# شكل الداتا دي
print("Shape of rows with less than 3 words:", short_rows.shape)

# عرض الأعمدة المهمة
display(short_rows[['Body', 'word_count','Department']].head(20))

In [ ]:
pd.set_option('display.max_colwidth', None)

display(short_rows[['Body', 'word_count','Department']])

كلمة system
توديك ع قسم
Technical Support



## PII Placeholder Detection



**Reasoning**:
I will scan the 'Body' column for specific PII placeholders like 'acc_num', 'nametel_num', and '[Your Name]' to assess the existing data masking as instructed.



In [ ]:
import pandas as pd
import re
from collections import Counter

# 1. تجهيز النصوص والتأكد من عدم وجود قيم مفقودة
body_text = df['Body'].fillna('')

# 2. تعريف "شكل" الكلمات التي نبحث عنها (الأنماط - Patterns)
# النمط الأول: أي كلمة تحتوي على شرطة سفلية (_) بين الحروف
underscore_pattern = r'\b[a-zA-Z]+_[a-zA-Z]+\b'

# النمط الثاني: أي جملة أو كلمة مكتوبة داخل أقواس مربعة [ ]
bracket_pattern = r'\[.*?\]'

# 3. استخراج هذه الأشكال من جميع التذاكر
discovered_placeholders = []

for text in body_text:
    # استخراج الكلمات ذات الشرطة السفلية
    discovered_placeholders.extend(re.findall(underscore_pattern, text))
    # استخراج الكلمات التي بداخل أقواس
    discovered_placeholders.extend(re.findall(bracket_pattern, text))

# 4. حساب عدد التكرارات لكل كلمة تم اكتشافها
placeholder_counts = Counter(discovered_placeholders)

# 5. طباعة النتائج (أكثر 10 كلمات بديلة تم اكتشافها)
print('--- Automatically Detected PII & Placeholders ---')
for placeholder, count in placeholder_counts.most_common(10):
    print(f'Placeholder "{placeholder}": {count} occurrences')

# حساب النسبة المئوية للتذاكر التي تحتوي على أي من هذه الأنماط
# نستخدم دالة لاختبار ما إذا كان النص يحتوي على أي من النمطين
combined_pattern = f'({underscore_pattern}|{bracket_pattern})'
total_with_pii = body_text.str.contains(combined_pattern, regex=True).sum()

print(f'\nTotal tickets with at least one placeholder: {total_with_pii}')
print(f'Percentage of dataset with placeholders: {(total_with_pii / len(df)) * 100:.2f}%')

دا بيحسب الكلمات المتكرره ف كل تيكت ومدي التكرار فيها


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter

all_words = " ".join(df['Body'].dropna()).split()
word_counts = Counter(all_words)

print(word_counts.most_common(20))
# حساب الكلمات
all_words = " ".join(df['Body'].dropna()).split()
word_counts = Counter(all_words)

# تحويل لأهم 20 كلمة
top_words = word_counts.most_common(20)
word_df = pd.DataFrame(top_words, columns=['Word', 'Frequency'])

# ترتيب عشان الشكل يبقى مظبوط
word_df = word_df.sort_values(by='Frequency', ascending=True)

# =========================
# الرسم
# =========================
plt.figure(figsize=(12, 8), dpi=300)

sns.barplot(
    data=word_df,
    x='Frequency',
    y='Word'
)

# Titles
plt.title('Top 20 Most Frequent Words in Tickets', fontsize=16, fontweight='bold')
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Word', fontsize=12)

# Add values on bars
for i, v in enumerate(word_df['Frequency']):
    plt.text(v + 10, i, str(v), va='center')

# تحسين الشكل
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

# حفظ الصورة
plt.savefig('word_frequency.png', dpi=300, bbox_inches='tight')

plt.show()

### Top 20 Keywords (After Removing Stopwords)

The previous analysis showed mostly stopwords (the, to, and...). Here we filter them out to see the actually meaningful words.

In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

all_words = " ".join(df['Body'].dropna()).lower().split()
# Filter out stopwords and very short words
meaningful_words = [w for w in all_words if w not in ENGLISH_STOP_WORDS and len(w) > 2]
word_counts_filtered = Counter(meaningful_words)

top_meaningful = word_counts_filtered.most_common(20)
meaningful_df = pd.DataFrame(top_meaningful, columns=['Word', 'Frequency'])
meaningful_df = meaningful_df.sort_values(by='Frequency', ascending=True)

plt.figure(figsize=(12, 8), dpi=150)
sns.barplot(data=meaningful_df, x='Frequency', y='Word')
plt.title('Top 20 Most Frequent MEANINGFUL Words (Stopwords Removed)', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nThese are the words that actually carry meaning for classification.")

## Update EDA Summary Report

### Subtask:
Update the final EDA summary report in cell 4493b03c with text length statistics and PII placeholder findings.


**Reasoning**:
I will create a code cell to update the `eda_summary` string by incorporating text length statistics and PII placeholder findings from the previous analysis steps.



**Top Keywords per Department**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# ناخد النصوص
df_clean = df[['Body', 'Department']].dropna()

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df_clean['Body'])

feature_names = vectorizer.get_feature_names_out()

# نحول لـ DataFrame
tfidf_df = pd.DataFrame(X.toarray(), columns=feature_names)
tfidf_df['Department'] = df_clean['Department'].values

# نجيب أهم كلمات لكل Department
top_words_per_dept = {}

for dept in tfidf_df['Department'].unique():
    subset = tfidf_df[tfidf_df['Department'] == dept]
    mean_scores = subset.drop(columns=['Department']).mean()
    top_words = mean_scores.sort_values(ascending=False).head(10)
    top_words_per_dept[dept] = top_words

# عرض النتائج
for dept, words in top_words_per_dept.items():
    print(f"\n=== {dept} ===")
    print(words)

اشوف معني كلمه data ليه بتكرر باشكل

In [ ]:
import pandas as pd
import re
from collections import Counter

# ══════════════════════════════════════════════════════════════
# Analysis: Why does "data" appear as top keyword in ALL departments?
# ══════════════════════════════════════════════════════════════

bodies = df['Body'].dropna().str.lower()

# ─── 1. How common is "data"? ───
total_with_data = bodies.str.contains(r'\bdata\b').sum()
print("=" * 65)
print(f'  "data" appears in {total_with_data} / {len(bodies)} tickets ({total_with_data/len(bodies)*100:.1f}%)')
print("=" * 65)

# ─── 2. "data" frequency per department ───
print("\n── Frequency of 'data' per Department ──\n")
print(f"  {'Department':<40} {'Has data':>10} {'Total':>8} {'%':>8}")
print("  " + "─" * 68)

for dept in df['Department'].value_counts().index:
    subset = df[df['Department'] == dept]['Body'].dropna().str.lower()
    has = subset.str.contains(r'\bdata\b').sum()
    pct = has / len(subset) * 100
    bar = "█" * int(pct / 2)
    print(f"  {dept:<40} {has:>10} {len(subset):>8} {pct:>7.1f}%  {bar}")

# ─── 3. Context: What words come with "data"? ───
after = []
before = []
for body in bodies:
    after.extend(re.findall(r'\bdata\s+(\w+)', body))
    before.extend(re.findall(r'(\w+)\s+data\b', body))

# filter out stopwords from "before" list
stops = {'the', 'a', 'an', 'in', 'of', 'and', 'with', 'on', 'to', 'for', 'our', 'my', 'is', 'this', 's'}
before_clean = [w for w in before if w not in stops]

print("\n── Most Common Phrases: 'data ___' ──\n")
for word, count in Counter(after).most_common(10):
    print(f'  "data {word}"  →  {count} times')

print("\n── Most Common Phrases: '___ data' ──\n")
for word, count in Counter(before_clean).most_common(10):
    print(f'  "{word} data"  →  {count} times')

# ─── 4. Show real samples per department ───
print("\n" + "=" * 65)
print("  REAL SAMPLES: How 'data' is used in each department")
print("=" * 65)

for dept in df['Department'].value_counts().index:
    subset = df[df['Department'] == dept]
    has_data = subset[subset['Body'].str.contains(r'\bdata\b', case=False, na=False)]

    if len(has_data) == 0:
        continue

    samples = has_data.sample(min(2, len(has_data)), random_state=42)

    print(f"\n┌─── {dept} ───")
    for _, row in samples.iterrows():
        sentences = re.split(r'[.!?]', row['Body'])
        data_sent = [s.strip() for s in sentences if re.search(r'\bdata\b', s, re.IGNORECASE)]
        if data_sent:
            sent = data_sent[0][:150]
            # highlight
            sent = re.sub(r'\b(data)\b', r'**\1**', sent, flags=re.IGNORECASE)
            print(f"│  → \"{sent}\"")
    print("└" + "─" * 60)



كلمة "data" لوحدها = ضوضاء، أيوه. لأنها بتظهر في كل الأقسام بنفس النسبة فمش بتساعد الموديل يفرّق بينهم.
لكن كجزء من عبارة زي "data breach" أو "data analytics" = مش ضوضاء. دي إشارات تصنيف قوية.
فهي الاتنين في نفس الوقت، حسب السياق. وعشان كده سبناها، لأن لو شلناها هنخسر العبارات المفيدة. والـ TF-IDF بيتعامل مع الجزء بتاعها كضوضاء لوحده من غير ما نتدخل.

خلاصة موضوع كلمة "data"
النقطة الأولى: هي كلمة مجال مش كلمة تصنيف
كلمة "data" موجودة في 42% من التذاكر وموزعة بالتساوي على كل الأقسام. فهي زي كلمة "patient" في مستشفى. موجودة في كل حتة لأن المجال كله بيتكلم عنها.

النقطة التانية: مشلناهاش وده القرار الصح
سبناها لأن لوحدها الـ TF-IDF بيديها وزن قليل تلقائياً. وكجزء من عبارات زي "data breach" و "data analytics" بتبقى إشارة تصنيف قوية.

النقطة التالتة: ده علمنا حاجة مهمة عن الداتاسيت
إن التذاكر بتتشارك في كلمات كتير لأن المجال واحد. والتحدي مش في الكلمات المشتركة، التحدي في إن الموديل يفهم السياق. وده بالظبط اللي بيبرر المقارنة بين Classical ML و LLM في مشروعك.

تكتب إيه في البحث؟
جملتين في فصل الـ EDA وجملتين في الـ Discussion وخلاص:

في الـ EDA تكتب:

The word "data" appeared in 42% of all tickets across all departments with similar frequency (30-45%), indicating it is a domain-level term rather than a classification signal. This highlights the need for context-aware features such as bigrams rather than relying on individual words.

في الـ Discussion تكتب:

The prevalence of shared vocabulary across departments, exemplified by the word "data", explains why unigram-based models struggle with this dataset. This further justifies the use of instruction-tuned LLMs which can capture full contextual meaning beyond individual word frequencies.

خلاص كده
الموضوع ده خلص. كلمة "data" مش مشكلة ومش محتاجة علاج. هي مجرد ملاحظة بتقوي تحليلك وبتبرر اختياراتك في المشروع.



In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ══════════════════════════════════════════════════════════════
# Vocabulary Size Analysis
# ══════════════════════════════════════════════════════════════

# Vocabulary BEFORE stopwords
vec_raw = CountVectorizer()
vec_raw.fit(df['Body'].dropna())

vocab_before = len(vec_raw.vocabulary_)

# Vocabulary AFTER stopwords
vec_clean = CountVectorizer(stop_words='english')
vec_clean.fit(df['Body'].dropna())

vocab_after = len(vec_clean.vocabulary_)

print(f"Vocabulary BEFORE stopword removal: {vocab_before:,}")
print(f"Vocabulary AFTER stopword removal:  {vocab_after:,}")
print(f"Stopwords removed: {vocab_before - vocab_after:,}")

# ══════════════════════════════════════════════════════════════
# Raw Frequency (WITH stopwords)
# ══════════════════════════════════════════════════════════════

all_words_raw = " ".join(df['Body'].dropna()).lower().split()

raw_counts = Counter(all_words_raw)

top_raw = pd.DataFrame(
    raw_counts.most_common(20),
    columns=['Word', 'Freq']
).sort_values('Freq', ascending=True)

# ══════════════════════════════════════════════════════════════
# Meaningful Frequency (WITHOUT stopwords)
# ══════════════════════════════════════════════════════════════

meaningful_words = [
    w for w in all_words_raw
    if w not in ENGLISH_STOP_WORDS and len(w) > 2
]

meaningful_counts = Counter(meaningful_words)

meaningful_df = pd.DataFrame(
    meaningful_counts.most_common(20),
    columns=['Word', 'Frequency']
)

meaningful_df_plot = meaningful_df.sort_values(
    'Frequency',
    ascending=True
)

# ══════════════════════════════════════════════════════════════
# Side-by-Side Visualization
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 8), dpi=150)

# Left plot
sns.barplot(
    data=top_raw,
    x='Freq',
    y='Word',
    hue='Word',
    legend=False,
    ax=axes[0],
)

axes[0].set_title(
    'Top 20 Words — Including Stopwords',
    fontweight='bold'
)

axes[0].set_xlabel('Frequency')

# Right plot
sns.barplot(
    data=meaningful_df_plot,
    x='Frequency',
    y='Word',
    hue='Word',
    legend=False,
    ax=axes[1],
)

axes[1].set_title(
    'Top 20 Words — After Stopword Removal',
    fontweight='bold'
)

axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('')

plt.suptitle(
    'Effect of Stopword Removal on Corpus Vocabulary',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    'freq_before_after_stopwords.png',
    dpi=150,
    bbox_inches='tight'
)

plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# ─── Step 1: Re-build TF-IDF ───
df_clean = df[['Body', 'Department']].dropna().copy()

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(df_clean['Body'])
feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(X.toarray(), columns=feature_names)
tfidf_df['Department'] = df_clean['Department'].values

# ─── Step 2: Compute similarity ───
dept_vectors = tfidf_df.groupby('Department').mean()
similarity_matrix = cosine_similarity(dept_vectors)
sim_df = pd.DataFrame(
    similarity_matrix,
    index=dept_vectors.index,
    columns=dept_vectors.index
)

print("TF-IDF similarity matrix built successfully.")
print(sim_df.round(3).to_string())

# ─── Step 3: Plot heatmap ───
fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

sns.heatmap(
    sim_df,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Cosine Similarity'},
    annot_kws={"size": 10}
)

ax.set_title("TF-IDF Cosine Similarity Between Departments",
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig('tfidf_similarity_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: tfidf_similarity_heatmap.png")

### 🔍 Keyword-Based Analysis per Department

The TF-IDF analysis reveals that each department is associated with a distinct set of high-importance keywords, indicating that the textual content of support tickets contains meaningful signals for classification.

#### Key Observations:

1. **Distinct Department Signatures**  
   Each department exhibits a unique vocabulary:
   - *Billing and Payments*: dominated by terms such as "billing", "payment", and "details".
   - *Technical / IT Support*: characterized by "issue", "software", "problem", and "server".
   - *Service Outages and Maintenance*: strongly associated with "outage", "network", and "service".

   This suggests that the classification task is feasible using textual features.

2. **Presence of Common Words Across Departments**  
   Several terms such as "data", "issue", and "support" appear across multiple departments.
   These words are less informative for classification and may introduce noise.
دي تحتاج preprossing
3. **Overlap Between Certain Departments**  
   Some departments (e.g., IT Support vs Technical Support, Customer Service vs General Inquiry) share similar keywords, which may lead to misclassification.

4. **Clearly Separable Categories**  
   Departments such as Billing and Service Outages show highly distinctive keywords, indicating that they are easier to classify.

#### Conclusion:

The TF-IDF analysis confirms that ticket text contains sufficient discriminative features for machine learning classification. However, additional preprocessing will be necessary to reduce noise from generic terms and improve model performance.

### 🔍 تحليل التشابه بين الأقسام (Department Similarity Analysis)

في هذا الجزء، قمنا بتحليل مدى التشابه بين الأقسام المختلفة بناءً على النصوص الموجودة في تذاكر الدعم الفني.

#### 🧠 خطوات العمل:

1. **تحويل النص إلى تمثيل رقمي (TF-IDF)**  
   تم تحويل كل تذكرة إلى مجموعة من القيم الرقمية التي تعبّر عن أهمية الكلمات داخل النص.

2. **تمثيل كل قسم بمتوسط القيم**  
   تم تجميع التذاكر حسب كل قسم (Department)، ثم حساب متوسط قيمة TF-IDF لكل كلمة.  
   وبهذا أصبح لكل قسم تمثيل رقمي يعكس "أسلوبه اللغوي".

3. **حساب التشابه بين الأقسام**  
   استخدمنا خوارزمية *Cosine Similarity* لمقارنة الأقسام ببعض ومعرفة مدى التشابه في استخدام الكلمات.

4. **عرض النتائج باستخدام Heatmap**  
   تم عرض النتائج في شكل خريطة حرارية (Heatmap)، حيث:
   - القيم القريبة من **1** تعني أن الأقسام متشابهة جدًا  
   - القيم القريبة من **0** تعني أن الأقسام مختلفة  

#### 🎯 الهدف من التحليل:

- فهم العلاقة بين الأقسام المختلفة  
- تحديد الأقسام المتشابهة لغويًا  
- توقع التحديات المحتملة في عملية التصنيف  

### 🔍 Department Similarity Analysis

In this section, we analyze the similarity between different departments based on the textual content of support tickets.

#### 🧠 Methodology:

1. **Text Representation using TF-IDF**  
   Each ticket is converted into a numerical vector representing the importance of words in the text.

2. **Department-Level Representation**  
   Tickets are grouped by their department, and the average TF-IDF score is computed for each word.  
   This results in a single vector representing the overall linguistic pattern of each department.

3. **Similarity Computation**  
   We use *Cosine Similarity* to measure how similar departments are based on their textual representations.

4. **Visualization using Heatmap**  
   The results are visualized using a heatmap:
   - Values close to **1** indicate high similarity  
   - Values close to **0** indicate low similarity  

#### 🎯 Purpose:

- To understand relationships between departments  
- To identify overlapping categories  
- To anticipate potential classification challenges  

###  Interpretation of Department Similarity Heatmap

The similarity heatmap reveals several important insights about the relationships between departments:

1. **High Overall Similarity**  
   Most departments exhibit high similarity scores (above 0.8), indicating that many tickets share common vocabulary across different categories.

2. **Strong Overlap Between Technical Departments**  
   IT Support and Technical Support show extremely high similarity (0.98), suggesting that they use nearly identical language and may be difficult to distinguish.

3. **Overlap Between Service-Oriented Departments**  
   Customer Service, General Inquiry, and Product Support also demonstrate high similarity, indicating potential classification ambiguity.

4. **Distinct Categories**  
   Billing and Payments shows relatively lower similarity with other departments, making it easier to classify.

5. **Moderately Distinct Category**  
   Service Outages and Maintenance appears somewhat distinct but still shares similarities with technical departments.

#### Conclusion:

The high similarity between many departments suggests that the classification task is challenging, especially for closely related categories. This insight highlights the importance of robust modeling and evaluation strategies.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Semantic Similarity Between Departments (using Sentence-BERT)
# ══════════════════════════════════════════════════════════════

# --- Step 0: Install (run once) ---
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ─── Step 1: Prepare text (remove stopwords first, as the doctor requested) ───

df_sem = df[['Body', 'Department']].dropna().copy()

def clean_for_semantic(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)           # HTML
    text = re.sub(r'http\S+', ' ', text)            # URLs
    text = re.sub(r'\[.*?\]', ' ', text)            # [Your Name]
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)        # special chars
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS and len(w) > 2]
    return " ".join(words)

df_sem['clean'] = df_sem['Body'].apply(clean_for_semantic)

print("Step 1 Done: Text cleaned + stopwords removed")
print(f"Sample: {df_sem['clean'].iloc[0][:100]}...")

# ─── Step 2: Create one representative text per department ───
# We concatenate a random sample of tickets per department
# (using all tickets would be too slow)

dept_texts = {}
for dept in df_sem['Department'].unique():
    subset = df_sem[df_sem['Department'] == dept]['clean']
    # Sample up to 500 tickets per department for speed
    sample = subset.sample(min(500, len(subset)), random_state=42)
    dept_texts[dept] = " ".join(sample.values)

print(f"\nStep 2 Done: Created representative text for {len(dept_texts)} departments")

# ─── Step 3: Generate Semantic Embeddings using Sentence-BERT ───

model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight, fast, good quality

# For long texts, we split into chunks and average
def get_embedding(text, model, max_length=500):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_length):
        chunk = " ".join(words[i:i+max_length])
        chunks.append(chunk)
    embeddings = model.encode(chunks)
    return np.mean(embeddings, axis=0)

dept_names = sorted(dept_texts.keys())
dept_embeddings = []

for dept in dept_names:
    emb = get_embedding(dept_texts[dept], model)
    dept_embeddings.append(emb)
    print(f"  Encoded: {dept}")

dept_embeddings = np.array(dept_embeddings)
print(f"\nStep 3 Done: Generated semantic embeddings (shape: {dept_embeddings.shape})")

# ─── Step 4: Calculate Semantic Cosine Similarity ───

sem_similarity = cosine_similarity(dept_embeddings)
sem_df = pd.DataFrame(sem_similarity, index=dept_names, columns=dept_names)

print("\n── Semantic Similarity Matrix ──\n")
print(sem_df.round(3).to_string())

# ─── Step 5: Visualize ───

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left: Semantic Similarity
sns.heatmap(
    sem_df, annot=True, fmt=".2f", cmap="YlGnBu",
    linewidths=0.5, ax=axes[0],
    cbar_kws={'label': 'Cosine Similarity'}
)
axes[0].set_title("Semantic Similarity (Sentence-BERT)\nBased on MEANING",
                   fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Right: TF-IDF Similarity (for comparison, reusing earlier sim_df if available)
try:
    sns.heatmap(
        sim_df, annot=True, fmt=".2f", cmap="YlOrRd",
        linewidths=0.5, ax=axes[1],
        cbar_kws={'label': 'Cosine Similarity'}
    )
    axes[1].set_title("TF-IDF Similarity (Word-based)\nBased on SHARED WORDS",
                       fontsize=14, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
except:
    axes[1].text(0.5, 0.5, 'TF-IDF similarity not available\nRun TF-IDF analysis first',
                 ha='center', va='center', fontsize=14)
    axes[1].set_title("TF-IDF Similarity (not available)")

plt.tight_layout()
plt.savefig('semantic_vs_tfidf_similarity.png', dpi=300, bbox_inches='tight')
plt.show()

# ─── Step 6: Key Differences ───

print("\n" + "=" * 65)
print("  KEY COMPARISON: Semantic vs TF-IDF Similarity")
print("=" * 65)

try:
    diff = sem_df.values - sim_df.values
    diff_df = pd.DataFrame(diff, index=dept_names, columns=dept_names)

    # Find biggest differences
    pairs = []
    for i in range(len(dept_names)):
        for j in range(i+1, len(dept_names)):
            pairs.append({
                'Dept A': dept_names[i],
                'Dept B': dept_names[j],
                'TF-IDF Sim': sim_df.iloc[i, j],
                'Semantic Sim': sem_df.iloc[i, j],
                'Difference': sem_df.iloc[i, j] - sim_df.iloc[i, j]
            })

    pairs_df = pd.DataFrame(pairs).sort_values('Difference', key=abs, ascending=False)

    print("\nPairs with BIGGEST difference between Semantic and TF-IDF:\n")
    print(pairs_df.head(10).to_string(index=False))

    print("\n── Interpretation ──")
    print("• Positive difference = Semantic finds MORE similarity than TF-IDF")
    print("  (departments use different words but talk about similar topics)")
    print("• Negative difference = Semantic finds LESS similarity than TF-IDF")
    print("  (departments share words but have different meanings)")
except:
    print("\nCannot compare — run TF-IDF similarity analysis first")

In [ ]:
# ─── Save Semantic heatmap ALONE ───
fig_sem, ax_sem = plt.subplots(figsize=(12, 10), dpi=300)

sns.heatmap(
    sem_df,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    linewidths=0.5,
    ax=ax_sem,
    cbar_kws={'label': 'Cosine Similarity'},
    annot_kws={"size": 10}
)

ax_sem.set_title("Semantic Similarity Between Departments\n(Sentence-BERT: all-MiniLM-L6-v2)",
                  fontsize=14, fontweight='bold', pad=20)

# Fix x-axis labels
ax_sem.set_xticklabels(
    ax_sem.get_xticklabels(),
    rotation=45,
    ha='right',
    fontsize=9
)

# Fix y-axis labels
ax_sem.set_yticklabels(
    ax_sem.get_yticklabels(),
    rotation=0,
    fontsize=9
)

plt.tight_layout()
plt.savefig('semantic_similarity_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: semantic_similarity_heatmap.png")


أول حاجة: إيه اللي بنشوفه؟
الخريطة اليمين (TF-IDF) بتقارن الأقسام على أساس الكلمات المشتركة. الخريطة الشمال (Semantic) بتقارن على أساس المعنى.

ثانياً: إيه أهم الملاحظات؟
ملاحظة 1: الـ Semantic Similarity أعلى من الـ TF-IDF في أغلب الحالات.
مثلاً Customer Service و General Inquiry: في الـ TF-IDF التشابه 0.91 بس في الـ Semantic طلع 0.98. ده معناه إن القسمين دول مش بس بيستخدموا نفس الكلمات، لا ده كمان المعنى الفعلي للتذاكر بتاعتهم قريب جداً من بعض. يعني التذاكر فعلاً بتتكلم عن نفس المواضيع.
ملاحظة 2: قسم Billing and Payments و Service Outages هم الأكتر اختلافاً عن باقي الأقسام.
في الـ Semantic، الـ Billing مع Service Outages عندهم 0.81 وده أقل رقم. وفي الـ TF-IDF الفرق أوضح: 0.54 بس. ده معناه إن القسمين دول عندهم لغة مميزة ومواضيع مختلفة عن الباقي. الموديل هيقدر يصنفهم بسهولة.
ملاحظة 3: فيه مجموعة أقسام متشابهة جداً ودي هتكون مشكلة.
Customer Service و General Inquiry و Product Support و Returns and Exchanges كلهم عندهم semantic similarity فوق 0.97. ده معناه إن التذاكر بتاعتهم بتتكلم عن مواضيع شبه بعض فعلاً. الموديل هيلاقي صعوبة كبيرة يفرّق بينهم.

ثالثاً: ده بيوصلك لإيه في المشروع؟
بالنسبة للـ Classical ML (Approach A):
الأقسام اللي التشابه بينها عالي (فوق 0.95) زي Customer Service مع General Inquiry هتكون أصعب حاجة على الموديلات الكلاسيكية. متوقع إن الـ Confusion Matrix هيبين خلط كبير بينهم. ده مش عيب في الموديل بتاعك، ده طبيعة الداتا نفسها.
بالنسبة للـ Fine-Tuning (Approach B):
هنا الفكرة إن الـ LLM ممكن يكون أحسن في التفريق بين الأقسام المتشابهة دي لأنه بيفهم السياق الكامل للتذكرة مش بس الكلمات. وده بالظبط نقطة المقارنة في مشروعك.
بالنسبة للكتابة:
ده بيديك فقرة قوية في فصل الـ Discussion. تقدر تقول إن التحليل الـ Semantic أثبت إن بعض الأقسام متشابهة فعلاً في المعنى مش بس في الكلمات، وده بيفسر ليه الموديل بيغلط في أقسام معينة. وتربطها بنتائج الـ Confusion Matrix.

ثالثاً: إيه اللي تكتبه في البحث؟
دي نقطة ممتازة تحطها في فصل الـ Discussion. تقدر تكتب إن:
التحليل أثبت إن التشابه بين بعض الأقسام هو تشابه حقيقي في المعنى وليس مجرد تشابه في الكلمات. وبالتالي فإن تحسين الـ preprocessing لن يحل المشكلة لأن المشكلة في طبيعة البيانات نفسها. وهذا يبرر استخدام نماذج أكثر تطوراً مثل الـ instruction-tuned LLMs التي تفهم السياق الكامل للنص بدلاً من الاعتماد على الكلمات المنفردة.


***prepossing***

In [ ]:
df.isna().sum()

In [ ]:
import pandas as pd
import re

# نشتغل على نسخة
df_ml = df[['Body', 'Department']].copy()



**lowercase**

In [ ]:
df_ml['Body_clean_ml'] = df_ml['Body'].str.lower()

display(df_ml[['Body', 'Body_clean_ml']].head(3))

In [ ]:
print('--- Missing Values Count ---')
missing_values = df.isnull().sum()
print(missing_values)

print('\n--- Duplicated Rows Count ---')
duplicate_count = df.duplicated().sum()
print(f'Total duplicate rows: {duplicate_count}')

**HTML removal**

In [ ]:
html_pattern = r'<[^>]+>'

df_ml['Body_clean_ml'] = df_ml['Body_clean_ml'].str.replace(
    html_pattern, ' ', regex=True
)

display(df_ml[['Body', 'Body_clean_ml']].head(3))

In [ ]:
before_html = df_ml['Body'].str.contains(html_pattern, regex=True).sum()
after_html = df_ml['Body_clean_ml'].str.contains(html_pattern, regex=True).sum()

print("Rows with HTML before cleaning:", before_html)
print("Rows with HTML after cleaning:", after_html)

**url removal**

In [ ]:
url_pattern = r'http[s]?://\S+|www\.\S+'

df_ml['Body_clean_ml'] = df_ml['Body_clean_ml'].str.replace(
    url_pattern, ' ', regex=True
)

display(df_ml[['Body', 'Body_clean_ml']].head(3))

In [ ]:
before_url = df_ml['Body'].str.contains(url_pattern, regex=True).sum()
after_url = df_ml['Body_clean_ml'].str.contains(url_pattern, regex=True).sum()

print("Rows with URLs before cleaning:", before_url)
print("Rows with URLs after cleaning:", after_url)

**placeholders removal**


In [ ]:
import re

# === FIXED: Use specific placeholder list instead of broad underscore pattern ===
# The broad pattern r'\b[a-zA-Z]+_[a-zA-Z]+\b' can accidentally remove
# meaningful words like "real_time" or "customer_service".
# Instead, we use the specific placeholders discovered in the EDA.

known_placeholders = [
    'acc_num', 'tel_num', 'nametel_num', 'nameacc_num',
    'order_num', 'email_address', 'company_name',
    'acc_numPhone', 'comtel_num', 'numtel_num',
    'nameorder_num', 'addresstel_num'
]

bracket_pattern = r'\[.*?\]'

def remove_placeholders(text):
    # Remove [Your Name] and similar bracket placeholders
    text = re.sub(bracket_pattern, ' ', text)
    # Remove only known underscore placeholders
    for ph in known_placeholders:
        text = re.sub(r'\b' + re.escape(ph) + r'\b', ' ', text)
    return text

In [ ]:
df_ml['Body_clean_ml'] = df_ml['Body_clean_ml'].apply(remove_placeholders)

display(df_ml[['Body', 'Body_clean_ml']].head(5))

**boilerplate removal**

In [ ]:
# === Boilerplate Phrases ===
# Note: Removed standalone "regards" as it's too aggressive
# and could match inside meaningful sentences.
# Only using full phrases that are clearly boilerplate.

boilerplate_phrases = [
    "dear customer support team",
    "dear support team",
    "i hope this message finds you well",
    "i hope this message reaches you well",
    "thank you for your assistance",
    "thank you for your time and assistance",
    "thank you for your time",
    "thank you for your help",
    "kind regards",
    "best regards",
    "warm regards",
    "sincerely",
    "looking forward to your response",
    "i look forward to your reply",
    "could you please",
    "i would appreciate"
]

def remove_boilerplate(text):
    for phrase in boilerplate_phrases:
        text = re.sub(re.escape(phrase), ' ', text)
    return text

مهم

في مرحلة ML:

إزالة boilerplate مفيدة جدًا
لكن لازم نبدأ بـ القائمة الواضحة جدًا فقط
ما نتوسعش قوي من البداية، عشان ما نمسحش معلومات مهمة بالغلط

In [ ]:
df_ml['Body_clean_ml'] = df_ml['Body_clean_ml'].apply(remove_boilerplate)

display(df_ml[['Body', 'Body_clean_ml']].head(5))

**extra cleaning fter this removal**

In [ ]:
def normalize_text(text):
    # remove non-letters except spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_ml['Body_clean_ml'] = df_ml['Body_clean_ml'].apply(normalize_text)

display(df_ml[['Body', 'Body_clean_ml']].head(5))

In [ ]:
df_ml.isna().sum()


**stop words removal**

In [ ]:
# ══════════════════════════════════════════════════════════════
# Stopwords Removal + Impact Visualization
# ══════════════════════════════════════════════════════════════

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import matplotlib.pyplot as plt

# ─── 1. Before: word counts with stopwords (current state) ───
before_lengths = df_ml['Body_clean_ml'].str.split().str.len()

# ─── 2. Remove stopwords ───
def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df_ml['Body_clean_ml'] = df_ml['Body_clean_ml'].apply(remove_stopwords)

# ─── 3. After: word counts without stopwords ───
after_lengths = df_ml['Body_clean_ml'].str.split().str.len()

# ─── 4. Print the difference ───
print("=" * 55)
print("  IMPACT OF STOPWORD REMOVAL")
print("=" * 55)
print(f"  {'Metric':<30} {'Before':>8} {'After':>8} {'Change':>10}")
print("  " + "─" * 56)
print(f"  {'Mean word count':<30} {before_lengths.mean():>8.1f} {after_lengths.mean():>8.1f} {after_lengths.mean() - before_lengths.mean():>+10.1f}")
print(f"  {'Median word count':<30} {before_lengths.median():>8.1f} {after_lengths.median():>8.1f} {after_lengths.median() - before_lengths.median():>+10.1f}")
print(f"  {'Total words removed':<30} {'':<8} {'':<8} {(before_lengths.sum() - after_lengths.sum()):>+10,}")

pct_removed = (1 - after_lengths.sum() / before_lengths.sum()) * 100
print(f"  {'% of words removed':<30} {'':<8} {'':<8} {pct_removed:>9.1f}%")
print("=" * 55)

# ─── 5. Visualization ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(before_lengths, bins=50, color='salmon', alpha=0.7, label='Before (with stopwords)')
axes[0].hist(after_lengths, bins=50, color='steelblue', alpha=0.7, label='After (without stopwords)')
axes[0].set_title('Word Count Distribution: Before vs After CLEANING', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Boxplot
axes[1].boxplot([before_lengths, after_lengths], labels=['Before', 'After'])
axes[1].set_title('Word Count Boxplot: Before vs After CLEANING', fontweight='bold')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

# ─── 6. Sample to see the difference ───
print("\n── Sample: Before vs After ──\n")
idx = df_ml.index[0]
original = df[df.index == idx]['Body'].values[0]
cleaned = df_ml.loc[idx, 'Body_clean_ml']
print(f"ORIGINAL:\n  {original[:200]}...\n")
print(f"AFTER CLEANING + STOPWORD REMOVAL:\n  {cleaned[:200]}...")

In [ ]:
df_ml.isna().sum()


## قراءة الرسم

---

### إيه اللي بنشوفه؟

الرسم ده بيقارن طول التذاكر قبل وبعد كل عمليات التنظيف مع بعض (حذف الـ HTML والـ placeholders والـ boilerplate والـ stopwords).

**في الـ Histogram (الرسم الشمال):**

الأحمر (Before) كان معظم التذاكر طولها بين 50 و 100 كلمة. الأزرق (After) اتزحلق لليسار وبقى معظمها بين 15 و 50 كلمة. يعني التنظيف شال تقريباً نص الكلمات.

**في الـ Boxplot (الرسم اليمين):**

الـ Median نزل من حوالي 55 كلمة لـ 30 كلمة. الـ Box كله صغر وده معناه إن التذاكر بقت أكتر تجانساً في الطول. الـ Outliers نزلت من 350 كلمة لـ 190 تقريباً.

---

### ده معناه إيه؟

معناه إن حوالي 45% من كلمات كل تذكرة كانت حاجات مش مفيدة للتصنيف: كلمات شائعة زي "the, is, to" وعبارات جاهزة زي "dear customer support team" وعلامات HTML و placeholders. كل ده اتشال ودلوقتي النص اللي فاضل هو اللب: الكلمات اللي فعلاً بتوصف المشكلة وبتساعد الموديل يصنف.

---

### بيساعدني فين؟

**أولاً: في أداء الموديل.**

الموديل دلوقتي بيشوف كلمات مفيدة بس. بدل ما يحاول يلاقي إشارات تصنيف وسط 80 كلمة أغلبها ضوضاء، دلوقتي بيشوف 35 كلمة كلها مهمة. ده المفروض يحسن النتائج.

**ثانياً: في الكتابة.**

الرسم ده بيتحط في فصل الـ Preprocessing كدليل بصري إن التنظيف عمل فرق واضح. بتكتب جنبه:

Preprocessing reduced the average ticket length from approximately 55 words to 30 words, removing around 45% of the content. This reduction primarily eliminated stopwords, boilerplate phrases, HTML tags, and PII placeholders, retaining only the semantically meaningful terms relevant for classification.

**ثالثاً: في المقارنة بين الـ Approach A والـ B.**

ده بيبين إن الطريقة الكلاسيكية محتاجة شغل تنظيف كتير عشان تشتغل كويس. لكن في الـ Fine-Tuning مش هتعمل أي حاجة من ده لأن الـ LLM بيفهم النص كامل بالـ stopwords والـ boilerplate وكل حاجة. وده في حد ذاته نقطة مقارنة مهمة في مشروعك.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Lemmatization: Return each word to its base form
# ══════════════════════════════════════════════════════════════

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

# ─── Before Lemmatization: count vocabulary ───
vocab_before = set(" ".join(df_ml['Body_clean_ml']).split())

# ─── Apply Lemmatization ───
df_ml['Body_clean_ml'] = df_ml['Body_clean_ml'].apply(lemmatize_text)

# ─── After Lemmatization: count vocabulary ───
vocab_after = set(" ".join(df_ml['Body_clean_ml']).split())

# ─── Show Impact ───
print("=" * 55)
print("  IMPACT OF LEMMATIZATION")
print("=" * 55)
print(f"  Vocabulary BEFORE: {len(vocab_before):,} unique words")
print(f"  Vocabulary AFTER:  {len(vocab_after):,} unique words")
print(f"  Words merged:      {len(vocab_before) - len(vocab_after):,} ({(len(vocab_before) - len(vocab_after)) / len(vocab_before) * 100:.1f}%)")
print("=" * 55)

# ─── Show examples of what changed ───
print("\n── Examples of Lemmatized Words ──\n")

examples = [
    ("issues", lemmatizer.lemmatize("issues")),
    ("strategies", lemmatizer.lemmatize("strategies")),
    ("services", lemmatizer.lemmatize("services")),
    ("running", lemmatizer.lemmatize("running", pos='v')),
    ("payments", lemmatizer.lemmatize("payments")),
    ("devices", lemmatizer.lemmatize("devices")),
    ("companies", lemmatizer.lemmatize("companies")),
    ("outages", lemmatizer.lemmatize("outages")),
    ("vulnerabilities", lemmatizer.lemmatize("vulnerabilities")),
    ("integrations", lemmatizer.lemmatize("integrations")),
]

for before, after in examples:
    changed = "→" if before != after else "  (no change)"
    print(f'  {before:<25s} {changed} {after}')

# ─── Show a sample ticket before vs after ───
print("\n── Sample Ticket ──\n")
sample_idx = df_ml.index[5]
print(f"BEFORE lemmatization (first 150 chars):")
# We need to re-derive "before" from earlier steps, so just show current
print(f"  {df_ml.loc[sample_idx, 'Body_clean_ml'][:150]}...")

In [ ]:
df_ml.isna().sum()


In [ ]:
# ══════════════════════════════════════════════════════════════
# Impact of Lemmatization on TF-IDF Keywords
# ══════════════════════════════════════════════════════════════

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# ─── 1. Save current (after lemmatization) text ───
after_lemma = df_ml['Body_clean_ml'].copy()

# ─── 2. Quick reverse test: what would it look like WITHOUT lemmatization? ───
# We re-read the original and apply all steps EXCEPT lemmatization
df_test = df[['Body', 'Department']].dropna().copy()
df_test['text'] = df_test['Body'].str.lower()
df_test['text'] = df_test['text'].str.replace(r'<[^>]+>', ' ', regex=True)
df_test['text'] = df_test['text'].str.replace(r'http\S+', ' ', regex=True)
df_test['text'] = df_test['text'].apply(remove_placeholders)
df_test['text'] = df_test['text'].apply(remove_boilerplate)
df_test['text'] = df_test['text'].apply(normalize_text)
df_test['text'] = df_test['text'].apply(remove_stopwords)

before_lemma = df_test['text']

# ─── 3. TF-IDF on both ───
vec_before = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
vec_after  = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

vec_before.fit(before_lemma)
vec_after.fit(after_lemma)

vocab_before = set(vec_before.get_feature_names_out())
vocab_after  = set(vec_after.get_feature_names_out())

# ─── 4. Vocabulary comparison ───
print("=" * 55)
print("  VOCABULARY COMPARISON")
print("=" * 55)
print(f"  TF-IDF features BEFORE lemmatization: {len(vocab_before):,}")
print(f"  TF-IDF features AFTER lemmatization:  {len(vocab_after):,}")
print(f"  Reduction: {len(vocab_before) - len(vocab_after):,} features")
print("=" * 55)

# ─── 5. Show words that got merged ───
disappeared = vocab_before - vocab_after
appeared = vocab_after - vocab_before

print(f"\n── Words that DISAPPEARED (merged into base form): {len(disappeared)} ──\n")
sample_gone = sorted(list(disappeared))[:20]
for w in sample_gone:
    print(f"  ✗ {w}")

print(f"\n── Base forms that APPEARED (result of merging): {len(appeared)} ──\n")
sample_new = sorted(list(appeared))[:20]
for w in sample_new:
    print(f"  ✓ {w}")

# ─── 6. Top keywords comparison per department ───
print("\n" + "=" * 55)
print("  TOP KEYWORDS: BEFORE vs AFTER LEMMATIZATION")
print("=" * 55)

X_b = vec_before.transform(before_lemma)
X_a = vec_after.transform(after_lemma)

tfidf_b = pd.DataFrame(X_b.toarray(), columns=vec_before.get_feature_names_out())
tfidf_a = pd.DataFrame(X_a.toarray(), columns=vec_after.get_feature_names_out())

tfidf_b['Department'] = df_test['Department'].values
tfidf_a['Department'] = df_ml['Department'].values

for dept in ['Billing and Payments', 'Technical Support', 'IT Support']:
    top_b = tfidf_b[tfidf_b['Department'] == dept].drop(columns=['Department']).mean().sort_values(ascending=False).head(8)
    top_a = tfidf_a[tfidf_a['Department'] == dept].drop(columns=['Department']).mean().sort_values(ascending=False).head(8)

    print(f"\n┌─── {dept} ───")
    print(f"│")
    print(f"│  {'BEFORE lemmatization':<35s} {'AFTER lemmatization'}")
    print(f"│  {'─' * 35} {'─' * 35}")

    for i in range(8):
        bw = f"{top_b.index[i]:<20s} {top_b.values[i]:.4f}"
        aw = f"{top_a.index[i]:<20s} {top_a.values[i]:.4f}"
        print(f"│  {bw:<35s} {aw}")
    print("└" + "─" * 72)

أولاً: الـ Vocabulary

عدد الكلمات الكلي ما اتغيرش (5,000 كلمة)،
لأن الـ TfidfVectorizer محدد بـ max_features=5000.

لكن اللي حصل فعليًا:

1,315 كلمة اختفت
1,315 كلمة جديدة ظهرت مكانها

➡️ يعني تقريبًا ربع الـ vocabulary اتغير

ثانياً: الكلمات اللي اتغيرت

أمثلة واضحة:

"accounts" → "account"
"access issues" → "access issue"
"actions" → "action"
"services" → "service"
"details" → "detail"

➡️ الكلمات اللي كانت بأشكال مختلفة
✔️ اتوحدت في شكل واحد

وده بالظبط الهدف من الـ Lemmatization

ثالثاً: التأثير على الـ Keywords

خلينا ناخد مثال: قسم Billing and Payments

قبل Lemmatization:
"services" → 0.0257
"details" → 0.0290
بعد Lemmatization:
"service" → 0.0299 ⬆️
"detail" → 0.0290

➡️ "services" اتجمعت مع "service"
➡️ فـ التكرار زاد → والـ score زاد

كمان:

ظهرت كلمات جديدة زي "subscription" و "invoice"
➡️ لأن الـ vocabulary بقت أنضف وبتدي فرصة لكلمات أهم تظهر
رابعاً: أهم تغيير

في أقسام زي Technical Support و IT Support:

"issues" و "issue" كانوا منفصلين
➡️ دلوقتي اتوحدوا في "issue"

✔️ النتيجة:

"issue" بقت أعلى كلمة بدل ما كانت التانية
مثال تاني:
"tools" → "tool"
الـ score زاد من 0.0209 → 0.0229

➡️ لأن التكرارات اتجمعت في كلمة واحدة

🎯 الخلاصة
الـ Lemmatization اشتغلت بشكل صحيح
غيرت حوالي 25% من الـ vocabulary
وحدت الكلمات المتشابهة
رفعت scores الكلمات المهمة لأن التكرار اتجمع

➡️ وده طبيعي يؤدي إلى:

تحسين أداء الموديلات

In [ ]:
df_ml.isna().sum()


In [ ]:
# ══════════════════════════════════════════════════════════════
# Clean Dataset Before Saving
# ══════════════════════════════════════════════════════════════

import pandas as pd

# Remove missing values
df_ml = df_ml.dropna(subset=['Body_clean_ml'])

# Remove empty strings
df_ml = df_ml[df_ml['Body_clean_ml'].astype(str).str.strip() != '']

# Reset index
df_ml = df_ml.reset_index(drop=True)

print("Final cleaned shape:", df_ml.shape)

# Check missing values
print("\nMissing values:\n")
print(df_ml.isna().sum())

# Save cleaned dataset
save_path = '/content/drive/MyDrive/df_ml_cleaned.csv'

df_ml.to_csv(save_path, index=False)

print(f"\nCleaned data saved to: {save_path}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Load Cleaned Dataset from Google Drive
# ══════════════════════════════════════════════════════════════

import pandas as pd
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
# Path to cleaned dataset
load_path = '/content/drive/MyDrive/df_ml_cleaned.csv'

# Prevent empty strings from becoming NaN
df_ml = pd.read_csv(load_path, keep_default_na=False)

# Final safety cleaning
df_ml = df_ml[df_ml['Body_clean_ml'].astype(str).str.strip() != '']

# Reset index
df_ml = df_ml.reset_index(drop=True)

# Features and labels
X_text = df_ml['Body_clean_ml']
y = df_ml['Department']

# Dataset information
print("Number of samples:", len(X_text))
print("Number of classes:", y.nunique())

print("\nClass Distribution:\n")
print(y.value_counts())

print("\nMissing values after loading:\n")
print(df_ml.isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split

# === FIXED: 80/10/10 split instead of 80/20 ===
# Added validation set for hyperparameter tuning

# Step 1: 80% train, 20% temp
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Step 2: Split temp into 50/50 → 10% val + 10% test
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"Train size: {len(X_train_text)} ({len(X_train_text)/len(X_text)*100:.0f}%)")
print(f"Val size:   {len(X_val_text)} ({len(X_val_text)/len(X_text)*100:.0f}%)")
print(f"Test size:  {len(X_test_text)} ({len(X_test_text)/len(X_text)*100:.0f}%)")
print(f"\nClass distribution in test set:")
print(y_test.value_counts())

that counts the unique words and we will use it in modelling

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
temp_vec = CountVectorizer(stop_words='english')
temp_vec.fit(X_train_text)
vocab_size = len(temp_vec.vocabulary_)
print(f'Vocab size: {vocab_size}')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# === FIXED: ngram_range changed from (1,5) to (1,2) ===
# (1,5) creates millions of sparse features → model gets confused
# (1,2) is the standard in research (unigrams + bigrams)
# max_features increased from 5000 to 10000 for better coverage

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000, #because we have n grams
    ngram_range=(1, 2),
    min_df=2,              # ignore words appearing in only 1 document
    max_df=0.95            # ignore words appearing in >95% of documents
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_val_tfidf   = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_text)


print("X_train shape:", X_train_tfidf.shape)
print("X_test shape:", X_test_tfidf.shape)
print("X_val_tfidf shape:", X_val_tfidf.shape)
print(f"\nVocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

micaro
macro

navie bayes


# Hyperparameter tuning is performed using GridSearchCV with cross-validation.
# The search is conducted on the training set only to avoid data leakage.
# The main parameter tuned is the regularization strength (C),
# along with solver selection.
# Macro F1-score is used as the evaluation metric due to class imbalance.

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
import seaborn as sns
import matplotlib.pyplot as plt
# ───  Hyperparameter Search (on train only, 3-fold CV) ───
# Grid covers compatible solver/penalty combinations:
#   lbfgs  → L2 only (multinomial loss, good for dense problems)
#   liblinear → L1 or L2 (OvR for multiclass, fast on sparse)
#   saga   → L1 or L2 (multinomial loss, stochastic, slower)
# C: inverse regularisation strength; larger C = less penalty
# Scoring: macro F1 — equal weight to all 10 classes

lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

param_grid = [
    {'C': [0.01, 0.1, 1, 3, 10], 'solver': ['lbfgs'],     'penalty': ['l2']      },
    {'C': [0.01, 0.1, 1, 3, 10], 'solver': ['liblinear'], 'penalty': ['l1', 'l2']},
    {'C': [0.01, 0.1, 1, 3, 10], 'solver': ['saga'],      'penalty': ['l1', 'l2']},
]

grid = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

# ─── 3. Training with timing ───
train_start = time.time()
grid.fit(X_train_tfidf, y_train)
train_end = time.time()
tuning_time = train_end - train_start

print(f"\nBest parameters : {grid.best_params_}")
print(f"Best CV F1-macro: {grid.best_score_:.4f}")
print(f"GridSearch time : {tuning_time:.1f}s ({tuning_time/60:.2f} min)")

# ─── 4. Final model with best params ───
best_lr = grid.best_estimator_

# ─── 5. Evaluation with timing ───
# Validation
val_start = time.time()
val_pred  = best_lr.predict(X_val_tfidf)
val_end   = time.time()
val_time  = (val_end - val_start) / len(X_val_text) * 1000  # ms/sample

# Test (consulted once — final evaluation only)
test_start = time.time()
test_pred  = best_lr.predict(X_test_tfidf)
test_end   = time.time()
test_time  = (test_end - test_start) / len(X_test_text) * 1000  # ms/sample

# ─── 6. Results ───
print("\n" + "="*55)
print("  LOGISTIC REGRESSION — FINAL RESULTS")
print("="*55)
print(f"  Val  Accuracy : {accuracy_score(y_val, val_pred):.4f}")
print(f"  Val  F1-macro : {f1_score(y_val, val_pred, average='macro'):.4f}")
print(f"  Test Accuracy : {accuracy_score(y_test, test_pred):.4f}")
print(f"  Test F1-macro : {f1_score(y_test, test_pred, average='macro'):.4f}")
print(f"  Inference     : {test_time:.3f} ms/sample")
print("="*55)

print("\nClassification Report (Test):\n")
print(classification_report(y_test, test_pred))

# ─── 7. Confusion Matrix ───
cm = confusion_matrix(y_test, test_pred, labels=best_lr.classes_)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_lr.classes_,
            yticklabels=best_lr.classes_)
plt.title(f"Confusion Matrix — Logistic Regression (TF-IDF)\n"
          f"Best: C={grid.best_params_['C']}, "
          f"{grid.best_params_['penalty']}, {grid.best_params_['solver']}",
          fontsize=13, fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('lr_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

Fitting 3 folds for each of 25 candidates, totalling 75 fits

Best Parameters:
{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}

Best CV Score:
0.5256762371334004
واخد ساعه

# The best model obtained from GridSearchCV is evaluated on the validation set.
# This provides an additional check before final testing.
# It helps ensure that the model generalizes well to unseen data.

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
import seaborn as sns
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════
# LOGISTIC REGRESSION — Final Model (Best params from GridSearch)
# TF-IDF only — no metadata features
# ══════════════════════════════════════════════════════════════

# ─── 1. Define final model ───
# Best params from GridSearchCV: C=10, l2, liblinear
# C=10: low regularisation → model fits the high-dim TF-IDF space
# liblinear: coordinate-descent solver, efficient on sparse matrices
# class_weight='balanced': compensates for 21:1 class imbalance

best_lr = LogisticRegression(
    C=10,
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

# ─── 2. Train on TRAIN data (with timing) ───
train_start = time.time()
best_lr.fit(X_train_tfidf, y_train)
train_end   = time.time()
train_time  = train_end - train_start

print(f"Training time: {train_time:.2f}s")

# ─── 3. Validation evaluation ───
val_pred = best_lr.predict(X_val_tfidf)

print(f"\nValidation Accuracy : {accuracy_score(y_val, val_pred):.4f}")
print(f"Validation F1-macro : {f1_score(y_val, val_pred, average='macro'):.4f}")

# ─── 4. Test evaluation (consulted once — final only) ───
test_start = time.time()
test_pred  = best_lr.predict(X_test_tfidf)
test_end   = time.time()
test_time  = (test_end - test_start) / len(X_test_text) * 1000  # ms/sample

print(f"\nTest Accuracy       : {accuracy_score(y_test, test_pred):.4f}")
print(f"Test F1-macro       : {f1_score(y_test, test_pred, average='macro'):.4f}")
print(f"Inference latency   : {test_time:.3f} ms/sample")

print("\nClassification Report (Test):\n")
print(classification_report(y_test, test_pred))

# ─── 5. Confusion Matrix ───
cm     = confusion_matrix(y_test, test_pred, labels=best_lr.classes_)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_lr.classes_,
            yticklabels=best_lr.classes_)
plt.title("Confusion Matrix — Logistic Regression (TF-IDF)\n"
          "C=10 | L2 | liblinear | class_weight=balanced",
          fontsize=13, fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('lr_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

bow and naives bayes

In [ ]:
import time
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
import seaborn as sns
import matplotlib.pyplot as plt

# ── Baseline: MNB + BoW (default parameters, no tuning) ──
# Serves as a lower-bound reference for the tuned TF-IDF model.
# vocab_size = full unigram vocabulary after preprocessing (5,402 tokens)

bow_vectorizer = CountVectorizer(
    max_features=vocab_size,
    min_df=2,
    max_df=0.95
)

X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_val_bow   = bow_vectorizer.transform(X_val_text)
X_test_bow  = bow_vectorizer.transform(X_test_text)

train_start = time.time()
nb_baseline = MultinomialNB()           # alpha=1.0, fit_prior=True (defaults)
nb_baseline.fit(X_train_bow, y_train)
train_end   = time.time()

val_pred  = nb_baseline.predict(X_val_bow)
test_pred = nb_baseline.predict(X_test_bow)

print("="*50)
print("  MNB BASELINE (BoW, default params)")
print("="*50)
print(f"  Training time  : {train_end - train_start:.2f}s")
print(f"  Val  Accuracy  : {accuracy_score(y_val, val_pred):.4f}")
print(f"  Val  F1-macro  : {f1_score(y_val, val_pred, average='macro'):.4f}")
print(f"  Test Accuracy  : {accuracy_score(y_test, test_pred):.4f}")
print(f"  Test F1-macro  : {f1_score(y_test, test_pred, average='macro'):.4f}")
print("="*50)

print("\nClassification Report (Test):\n")
print(classification_report(y_test, test_pred))

cm = confusion_matrix(y_test, test_pred, labels=nb_baseline.classes_)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nb_baseline.classes_,
            yticklabels=nb_baseline.classes_)
plt.title("Confusion Matrix — MNB Baseline (BoW, default params)",
          fontsize=13, fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('nb_baseline_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

🧠 Hyperparameters في Naive Bayes (MultinomialNB)
✅ أولاً: أهم Hyperparameters
🔹 1) alpha (Smoothing Parameter) — الأهم

ده أهم باراميتر في الموديل.

💡 وظيفته:

بيحل مشكلة إن:

كلمة تظهر في الـ test ومش موجودة في الـ train
➡️ الاحتمال يبقى = 0 ❌

الـ alpha بيضيف قيمة صغيرة لكل الكلمات
➡️ علشان يمنع الصفر

⚖️ تأثير القيم:
alpha صغير (0.01 – 0.5):
حساس للكلمات النادرة
ممكن يحسن التمييز بين الأقسام
لكن ممكن يعمل overfitting
alpha كبير (5 – 10):
أكثر استقرارًا
يتجاهل الكلمات النادرة
ممكن يقلل الدقة في التمييز
📊 القيم اللي نجربها:
0.01, 0.1, 0.5, 1.0, 5.0, 10.0
🔹 2) fit_prior
💡 وظيفته:

هل الموديل:

يتعلم توزيع الأقسام من الداتا؟
ولا يفترضهم متساويين؟
القيم:
True → يتعلم من الداتا
False → يفترض تساوي الأقسام

➡️ مهم جدًا لأن الداتا غير متوازنة

🔹 3) class_prior
💡 وظيفته:

تحدد نسب الأقسام يدويًا

➡️ في حالتنا:

None

✔️ نخلي الموديل يتعلمها تلقائيًا

In [ ]:
import time
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# ══════════════════════════════════════════════════════════════
# MULTINOMIAL NAIVE BAYES — TF-IDF only (no metadata features)
# References:
#   Manning et al. (2008), Introduction to Information Retrieval,
#   Cambridge University Press, pp. 234–265.
#   McCallum & Nigam (1998), A Comparison of Event Models for
#   Naive Bayes Text Classification, AAAI Workshop.
# ══════════════════════════════════════════════════════════════

# ─── 1. Hyperparameter Grid ───
# MNB has exactly three tunable parameters:
#
#   alpha (Laplace/Lidstone smoothing):
#     Controls the additive constant added to all term counts before
#     normalisation. Small values (e.g. 0.01) preserve the influence
#     of rare, discriminative terms; large values (e.g. 10.0) flatten
#     term distributions toward uniformity, effectively ignoring rare
#     words. Manning et al. (2008) recommend alpha=1.0 as a principled
#     default; the optimal value is corpus-dependent.
#
#   fit_prior:
#     If True, the class prior P(c) is estimated from training-set
#     class frequencies. Under severe class imbalance (21:1 ratio in
#     this corpus), a learned prior amplifies the majority class and
#     may suppress minority-class recall. Setting fit_prior=False
#     enforces a uniform prior, delegating all discriminative work
#     to the likelihood term.
#
#   class_prior:
#     An alternative to fit_prior — allows manual specification of
#     class weights. None defers to fit_prior; a uniform vector
#     [1/10]*10 replicates fit_prior=False analytically.
#     Kept as None here to avoid redundancy with fit_prior=False.

param_grid = {
    'alpha':     [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    'fit_prior': [True, False]
}

# ─── 2. GridSearchCV (on train only — 3-fold, macro F1) ───
# cv=3 chosen for consistency with LR and RF experiments.
# scoring=f1_macro: assigns equal weight to all 10 classes,
# preventing majority-class dominance from inflating the score.

grid_search = GridSearchCV(
    estimator=MultinomialNB(),
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    return_train_score=True
)

train_start = time.time()
grid_search.fit(X_train_tfidf, y_train)
train_end   = time.time()
tuning_time = train_end - train_start

print(f"GridSearch time : {tuning_time:.1f}s ({tuning_time/60:.2f} min)")
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV F1-macro: {grid_search.best_score_:.4f}")

# ─── 3. All combinations (sorted) ───
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[[
    'param_alpha', 'param_fit_prior',
    'mean_test_score', 'std_test_score', 'rank_test_score'
]].sort_values('rank_test_score')
results_df.columns = ['alpha', 'fit_prior', 'Mean F1-Macro', 'Std', 'Rank']

print("\n── All combinations (sorted by CV F1-macro) ──\n")
print(results_df.to_string(index=False))

# ─── 4. Evaluate best model ───
best_nb = grid_search.best_estimator_

val_pred = best_nb.predict(X_val_tfidf)

val_start  = time.time()
test_pred  = best_nb.predict(X_test_tfidf)
test_end   = time.time()
test_time  = (test_end - val_start) / len(X_test_text) * 1000  # ms/sample

print("\n" + "="*55)
print("  MULTINOMIAL NAIVE BAYES — FINAL RESULTS")
print("="*55)
print(f"  Best alpha     : {grid_search.best_params_['alpha']}")
print(f"  Best fit_prior : {grid_search.best_params_['fit_prior']}")
print(f"  Val  Accuracy  : {accuracy_score(y_val, val_pred):.4f}")
print(f"  Val  F1-macro  : {f1_score(y_val, val_pred, average='macro'):.4f}")
print(f"  Test Accuracy  : {accuracy_score(y_test, test_pred):.4f}")
print(f"  Test F1-macro  : {f1_score(y_test, test_pred, average='macro'):.4f}")
print(f"  Inference      : {test_time:.3f} ms/sample")
print("="*55)

print("\nClassification Report (Test):\n")
print(classification_report(y_test, test_pred))

# ─── 5. Confusion Matrix ───
cm     = confusion_matrix(y_test, test_pred, labels=best_nb.classes_)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_nb.classes_,
            yticklabels=best_nb.classes_)
plt.title(
    f"Confusion Matrix — Multinomial Naive Bayes (TF-IDF)\n"
    f"alpha={grid_search.best_params_['alpha']} | "
    f"fit_prior={grid_search.best_params_['fit_prior']}",
    fontsize=13, fontweight='bold'
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('nb_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# ─── 6. Alpha sensitivity plot ───
best_prior = grid_search.best_params_['fit_prior']
alpha_df   = results_df[results_df['fit_prior'] == best_prior].copy()

plt.figure(figsize=(9, 5))
plt.plot(alpha_df['alpha'], alpha_df['Mean F1-Macro'],
         'bo-', linewidth=2, markersize=8)
plt.fill_between(
    alpha_df['alpha'],
    alpha_df['Mean F1-Macro'] - alpha_df['Std'],
    alpha_df['Mean F1-Macro'] + alpha_df['Std'],
    alpha=0.2
)
plt.xscale('log')
plt.xlabel('alpha (log scale)', fontsize=12)
plt.ylabel('CV F1-Macro', fontsize=12)
plt.title(f'Alpha Sensitivity — MNB (fit_prior={best_prior})',
          fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('nb_alpha_sensitivity.png', dpi=200, bbox_inches='tight')
plt.show()

🧠 📝 Arabic Version (للـ Notebook)
قراءة نتائج Naive Bayes
أفضل Hyperparameters

تم استخدام GridSearchCV لتجربة 12 تركيبة مختلفة من hyperparameters، وكانت أفضل القيم:

alpha = 0.01
fit_prior = False

تشير هذه النتيجة إلى أن النموذج يستفيد من القيم الصغيرة لـ alpha، حيث يصبح أكثر حساسية للكلمات النادرة، والتي تلعب دورًا مهمًا في التمييز بين الأقسام (مثل: "billing", "outage").

كما أن اختيار fit_prior = False يعني أن النموذج يعامل جميع الأقسام بالتساوي، وهو مناسب لأن البيانات غير متوازنة، حيث أن بعض الأقسام تمثل نسبة أكبر من البيانات.

تأثير معامل alpha

أظهرت النتائج أن قيمة alpha لها تأثير كبير على الأداء:

عند alpha = 0.01 → أفضل أداء (~0.49 F1)
مع زيادة alpha → الأداء ينخفض تدريجيًا
عند alpha = 10.0 → أداء ضعيف جدًا (~0.13 F1)

هذا يشير إلى أن الكلمات النادرة مهمة جدًا في هذه البيانات، وأن زيادة التنعيم (smoothing) تؤدي إلى فقدان هذه المعلومات المهمة.

مقارنة مع Logistic Regression

حقق نموذج Logistic Regression بعد الـ tuning دقة أعلى (64%) مقارنة بـ Naive Bayes (49%).

يرجع هذا الفرق إلى أن Naive Bayes يفترض استقلالية الكلمات، حيث يتعامل مع كل كلمة بشكل منفصل، بينما Logistic Regression يمكنه الاستفادة من العلاقات بين الكلمات، خاصة عند استخدام n-grams مثل "data breach".

تحليل الأداء حسب الأقسام

حقق Naive Bayes أداء جيدًا في بعض الأقسام مثل:

Billing and Payments (F1 ≈ 76%)
Human Resources (F1 ≈ 63%)

وذلك بسبب وجود كلمات مميزة وواضحة لهذه الأقسام.

في المقابل، كان الأداء أضعف في أقسام مثل:

Product Support
Sales and Pre-Sales

بسبب التشابه اللغوي بينها وعدم وجود كلمات مميزة كافية.

الخلاصة

تشير النتائج إلى أن Logistic Regression يتفوق على Naive Bayes في هذه المهمة، وذلك لأن البيانات تعتمد على علاقات بين الكلمات (مثل العبارات) وليس فقط على الكلمات الفردية.

كما تدعم هذه النتيجة استخدام نماذج أكثر تقدمًا قادرة على فهم السياق الكامل، مثل نماذج اللغة الكبيرة (LLMs).

🧠 📝 English Version (For Notebook / Report)
Naive Bayes Results Analysis
Best Hyperparameters

GridSearchCV was used to evaluate 12 different hyperparameter combinations. The best configuration was:

alpha = 0.01
fit_prior = False

This indicates that smaller values of alpha improve performance by making the model more sensitive to rare words, which are important for distinguishing between classes (e.g., "billing", "outage").

Setting fit_prior = False allows the model to treat all classes equally, which is beneficial for imbalanced datasets where some classes dominate.

Effect of Alpha

The results show a strong impact of the alpha parameter:

alpha = 0.01 → best performance (~0.49 F1)
Increasing alpha → performance decreases
alpha = 10.0 → very poor performance (~0.13 F1)

This demonstrates that rare words are highly informative in this dataset, and excessive smoothing reduces their impact.

Comparison with Logistic Regression

Logistic Regression achieved higher performance (64% accuracy) compared to Naive Bayes (49%).

This is because Naive Bayes assumes feature independence and treats each word separately, while Logistic Regression can capture relationships between words, especially when using n-grams such as "data breach".

Class-wise Performance Analysis

Naive Bayes performed well in:

Billing and Payments (F1 ≈ 76%)
Human Resources (F1 ≈ 63%)

due to the presence of distinctive keywords.

However, it performed poorly in:

Product Support
Sales and Pre-Sales

due to linguistic similarity and lack of distinctive features.

Conclusion

The results indicate that Logistic Regression outperforms Naive Bayes for this task, as the dataset relies on relationships between words (phrases) rather than individual tokens.

This also suggests that more advanced models capable of capturing contextual information, such as fine-tuned large language models, are expected to achieve better performance.

word 2 vec (word emding ) auf clean data




dann ML Random forest

In [ ]:
fpip install gensim

In [ ]:
# ══════════════════════════════════════════════════════════════
# PART 1: Word2Vec Representation
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from gensim.models import Word2Vec

print("=" * 60)
print("  PART 1: Training Word2Vec Model")
print("=" * 60)

# 1. Tokenization
train_tokens = X_train_text.apply(str.split).tolist()
val_tokens   = X_val_text.apply(str.split).tolist()
test_tokens  = X_test_text.apply(str.split).tolist()

print(f"Sample tokenized ticket: {train_tokens[0][:10]}...")

# 2. Train Word2Vec only on training data
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    seed=42,
    epochs=10
)

print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")
print(f"Vector size per word: {w2v_model.wv.vector_size}")

# 3. Document vector function
def document_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]

    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)

    return np.mean(vectors, axis=0)

# 4. Convert datasets into Word2Vec document vectors
X_train_w2v = np.array([document_vector(doc, w2v_model) for doc in train_tokens])
X_val_w2v   = np.array([document_vector(doc, w2v_model) for doc in val_tokens])
X_test_w2v  = np.array([document_vector(doc, w2v_model) for doc in test_tokens])

print(f"X_train_w2v shape: {X_train_w2v.shape}")
print(f"X_val_w2v shape:   {X_val_w2v.shape}")
print(f"X_test_w2v shape:  {X_test_w2v.shape}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# PART 2: Random Forest Hyperparameter Tuning + Training Time
# ══════════════════════════════════════════════════════════════

import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("  PART 2: Random Forest Hyperparameter Tuning")
print("=" * 60)

# 1. Parameter search space
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [30, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}

n_iter = 30

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=3,
    scoring='f1_macro',
    verbose=1,
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)

# 2. Hyperparameter tuning time
tuning_start_time = time.time()

random_search.fit(X_train_w2v, y_train)

tuning_end_time = time.time()
tuning_duration = tuning_end_time - tuning_start_time

print("\n" + "=" * 60)
print("  BEST PARAMETERS FOUND")
print("=" * 60)

for param, value in random_search.best_params_.items():
    print(f"{param:<25s} {value}")

print(f"\nBest CV F1-Macro: {random_search.best_score_:.4f}")

print("\n" + "=" * 60)
print("  HYPERPARAMETER TUNING TIME")
print("=" * 60)
print(f"Start time: {time.ctime(tuning_start_time)}")
print(f"End time:   {time.ctime(tuning_end_time)}")
print(f"Duration:   {tuning_duration / 60:.2f} minutes")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Train Final Random Forest using Best Hyperparameters
# ══════════════════════════════════════════════════════════════

best_params = random_search.best_params_

final_rf = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

actual_train_start = time.time()

final_rf.fit(X_train_w2v, y_train)

actual_train_end = time.time()
actual_train_duration = actual_train_end - actual_train_start

print("\n" + "=" * 60)
print("  FINAL MODEL ACTUAL TRAINING TIME")
print("=" * 60)
print(f"Training start: {time.ctime(actual_train_start)}")
print(f"Training end:   {time.ctime(actual_train_end)}")
print(f"Training time:  {actual_train_duration / 60:.2f} minutes")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════

val_pred  = final_rf.predict(X_val_w2v)
test_pred = final_rf.predict(X_test_w2v)

val_acc  = accuracy_score(y_val, val_pred)
test_acc = accuracy_score(y_test, test_pred)

print("\n" + "=" * 60)
print("  FINAL RESULTS: Word2Vec + Random Forest")
print("=" * 60)
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")

print("\nClassification Report - Test Set:\n")
print(classification_report(y_test, test_pred))

cm = confusion_matrix(y_test, test_pred)
labels = sorted(y_test.unique())

plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=labels,
    yticklabels=labels
)

plt.title(
    f'Word2Vec + Random Forest Confusion Matrix\nTest Accuracy: {test_acc:.4f}',
    fontsize=14,
    fontweight='bold'
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('w2v_rf_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

نهاية الجديد

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
from pathlib import Path

# غيّر الاسم ده لاسم النوتبوك بتاعك في Drive
path = Path("/content/drive/MyDrive/Colab Notebooks/YOUR_NOTEBOOK_NAME.ipynb")

nb = json.loads(path.read_text(encoding="utf-8"))

# حذف metadata.widgets المسببة لمشكلة GitHub
nb.get("metadata", {}).pop("widgets", None)

path.write_text(json.dumps(nb, indent=1, ensure_ascii=False), encoding="utf-8")

print("Fixed:", path)

**Fine tunning**

In [ ]:
# ═══════════════════════════════════════════════════
# Step 1.1: Install Unsloth and dependencies
# ═══════════════════════════════════════════════════
# Reference: https://github.com/unslothai/unsloth
# Reference: https://huggingface.co/blog/unsloth-trl

!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# ═══════════════════════════════════════════════════
# Step 1: Load model (FIXED: max_seq_length = 1024)
# ═══════════════════════════════════════════════════

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = 1024,    # ← FIXED: was 512
    dtype = None,
    load_in_4bit = True,
)

print("Model loaded successfully!")
print(f"Max sequence length: 1024 tokens")

In [ ]:
# ═══════════════════════════════════════════════════
# Step 2: Add LoRA
# ═══════════════════════════════════════════════════

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj",
                      "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

model.print_trainable_parameters()

first try

In [ ]:
# ═══════════════════════════════════════════════════
# Step 3: Prepare dataset (same style as reference notebook)
# ═══════════════════════════════════════════════════

import json
import os
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
# ─── 3.1 Load original data (NOT cleaned - LLMs need raw text) ───
df = pd.read_csv('/content/drive/MyDrive/IT Support Ticket Data.csv', index_col=0)
df = df.dropna(subset=['Body'])
df = df[['Body', 'Department']]

print(f"Total samples: {len(df)}")

# ─── 3.2 Train/Val/Test split (same random_state as Classical ML) ───
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['Department'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['Department'], random_state=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# ─── 3.3 Format data (same structure as reference notebook) ───
# Reference notebook uses: system, instruction, input, output, history
# We follow the exact same structure

system_message = "\n".join([
    "You are an IT support ticket classifier.",
    "Read the support ticket provided by the user.",
    "Classify it into the correct department.",
    "Only respond with the department name, nothing else.",
    "Do not generate any introduction or conclusion."
])

departments_list = sorted(df['Department'].unique().tolist())
departments_str = "\n".join([f"- {d}" for d in departments_list])

def format_sample(row):
    return {
        "system": system_message,
        "instruction": "\n".join([
            "# Support Ticket:",
            row['Body'],
            "",
            "# Available Departments:",
            departments_str,
            "",
            "# Department:"
        ]),
        "input": "",
        "output": row['Department'],
        "history": []
    }

# ─── 3.4 Convert all data ───
train_data = [format_sample(row) for _, row in train_df.iterrows()]
val_data   = [format_sample(row) for _, row in val_df.iterrows()]

# ─── 3.5 Show example ───
print("\n" + "=" * 60)
print("  EXAMPLE: Formatted training sample")
print("=" * 60)
print(f"\nSYSTEM:\n{train_data[0]['system']}")
print(f"\nINSTRUCTION:\n{train_data[0]['instruction'][:300]}...")
print(f"\nOUTPUT:\n{train_data[0]['output']}")

# ─── 3.6 Save as JSON (same as reference notebook) ───
save_dir = '/content/drive/MyDrive/ticket-classifier-data'
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, 'train.json'), 'w') as f:
    json.dump(train_data, f, ensure_ascii=False)

with open(os.path.join(save_dir, 'val.json'), 'w') as f:
    json.dump(val_data, f, ensure_ascii=False)

# Save test for later evaluation
test_df.to_csv(os.path.join(save_dir, 'test.csv'), index=False)

print(f"\nSaved to: {save_dir}")
print(f"  train.json: {len(train_data)} samples")
print(f"  val.json:   {len(val_data)} samples")
print(f"  test.csv:   {len(test_df)} samples")

In [ ]:
# ═══════════════════════════════════════════════════
# Step 4: Quick Test Training
# ═══════════════════════════════════════════════════

import torch
torch.cuda.empty_cache()

from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# ─── Load datasets ───
train_dataset = load_dataset('json',
    data_files='/content/drive/MyDrive/ticket-classifier-data/train.json',
    split='train'
)

train_mini = train_dataset.select(range(500))
print(f"Quick test: {len(train_mini)} samples")

# ─── Formatting function ───
def formatting_func(example):
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{example['system']}<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    return [text]

# ─── Training ───
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_mini,
    formatting_func = formatting_func,
    max_seq_length = 1024,
    packing = False,
    args = TrainingArguments(
        output_dir = "/content/test-run",
        num_train_epochs = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        learning_rate = 1e-4,
        logging_steps = 5,
        save_strategy = "no",
        eval_strategy = "no",
        fp16 = True,
        optim = "adamw_8bit",
        seed = 42,
        report_to = "none",
    ),
)

print("Training started...\n")
trainer.train()
print("\nDone!")

# ─── Quick test ───
FastLanguageModel.for_inference(model)

test_ticket = "My printer is not working and I need someone to fix it urgently."

messages = [
    {"role": "system", "content": "You are an IT support ticket classifier.\nClassify the ticket into the correct department.\nOnly respond with the department name."},
    {"role": "user", "content": f"# Support Ticket:\n{test_ticket}\n\n# Department:"}
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=20, do_sample=False)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print(f"\nTicket:    {test_ticket}")
print(f"Predicted: {response}")

second try

In [ ]:
# ═══════════════════════════════════════════════════
# Training on 3000 tickets + Full Evaluation
# ═══════════════════════════════════════════════════

import torch
torch.cuda.empty_cache()

from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# ─── 1. Load data ───
train_dataset = load_dataset('json',
    data_files='/content/drive/MyDrive/ticket-classifier-data/train.json',
    split='train'
)

train_3k = train_dataset.select(range(3000))
print(f"Training on: {len(train_3k)} samples")

# ─── 2. Formatting function ───
def formatting_func(example):
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{example['system']}<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    return [text]

# ─── 3. Train ───
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_3k,
    formatting_func = formatting_func,
    max_seq_length = 1024,
    packing = False,
    args = TrainingArguments(
        output_dir = "/content/train-3k",
        num_train_epochs = 3,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate = 1e-4,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.1,
        logging_steps = 50,
        save_strategy = "no",
        eval_strategy = "no",
        fp16 = True,
        optim = "adamw_8bit",
        seed = 42,
        report_to = "none",
    ),
)

print("Training started... (estimated 10-15 minutes)\n")
trainer.train()
print("\nTraining complete!")

# ─── 4. Evaluate on test set ───
FastLanguageModel.for_inference(model)

test_df = pd.read_csv('/content/drive/MyDrive/ticket-classifier-data/test.csv')
print(f"\nEvaluating on {len(test_df)} test tickets...")

y_true = []
y_pred = []

for i, row in test_df.iterrows():
    messages = [
        {"role": "system", "content": "You are an IT support ticket classifier.\nClassify the ticket into the correct department.\nOnly respond with the department name."},
        {"role": "user", "content": f"# Support Ticket:\n{row['Body']}\n\n# Department:"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=20, do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()

    y_true.append(row['Department'])
    y_pred.append(response)

    if (i + 1) % 500 == 0:
        print(f"  Evaluated {i+1}/{len(test_df)}...")

# ─── 5. Results ───
# Match predictions to valid departments
valid_depts = sorted(test_df['Department'].unique())

def match_department(pred):
    pred_lower = pred.lower().strip()
    for dept in valid_depts:
        if dept.lower() in pred_lower or pred_lower in dept.lower():
            return dept
    return pred

y_pred_matched = [match_department(p) for p in y_pred]

acc = accuracy_score(y_true, y_pred_matched)

print("\n" + "=" * 60)
print(f"  LLM FINE-TUNING RESULTS (3000 training samples)")
print(f"  Test Accuracy: {acc:.4f}")
print("=" * 60)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred_matched))

# ─── 6. Confusion Matrix ───
cm = confusion_matrix(y_true, y_pred_matched, labels=valid_depts)

plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=valid_depts, yticklabels=valid_depts)
plt.title(f'LLaMA 3.1 Fine-Tuned (3K samples)\nTest Accuracy: {acc:.4f}',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('llm_confusion_matrix_3k.png', dpi=200, bbox_inches='tight')
plt.show()

# ─── 7. Show some predictions ───
print("\n── Sample Predictions ──\n")
for i in range(10):
    status = "✓" if y_true[i] == y_pred_matched[i] else "✗"
    print(f"  {status} True: {y_true[i]:<35s} Pred: {y_pred_matched[i]}")

In [ ]:
# ═══════════════════════════════════════════════════
# FIXED: Proper training + constrained evaluation
# ═══════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from trl import SFTTrainer
from transformers import TrainingArguments

# ─── 1. Load and split data ───
df = pd.read_csv('/content/drive/MyDrive/IT Support Ticket Data.csv', index_col=0)
df = df.dropna(subset=['Body'])
df = df[['Body', 'Department']]

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['Department'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['Department'], random_state=42
)

# Take 3000 for training
train_df = train_df.head(3000)

DEPARTMENTS = sorted(df['Department'].unique().tolist())
dept_str = ", ".join(DEPARTMENTS)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Departments: {DEPARTMENTS}")

# ─── 2. Create Hugging Face Dataset with text column ───
# FIX: Instead of using formatting_func, we create the text directly

def make_text(row):
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an IT support ticket classifier.
You must classify tickets into one of these departments ONLY: {dept_str}
Respond with the department name only.<|eot_id|><|start_header_id|>user<|end_header_id|>

{row['Body']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{row['Department']}<|eot_id|>"""

train_texts = [make_text(row) for _, row in train_df.iterrows()]
train_dataset = Dataset.from_dict({"text": train_texts})

print(f"\nDataset created: {len(train_dataset)} samples")
print(f"Sample text (first 300 chars):\n{train_texts[0][:300]}...")

# ─── 3. Train ───
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    packing = False,
    args = TrainingArguments(
        output_dir = "/content/train-3k",
        num_train_epochs = 3,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate = 1e-4,
        lr_scheduler_type = "cosine",
        warmup_steps = 50,
        logging_steps = 50,
        save_strategy = "no",
        eval_strategy = "no",
        fp16 = True,
        optim = "adamw_8bit",
        seed = 42,
        report_to = "none",
    ),
)

print(f"\nTotal training steps: {trainer.state.max_steps}")
print("Training started...\n")
trainer.train()
print("\nTraining complete!")

In [ ]:
# ═══════════════════════════════════════════════════
# Evaluation with CONSTRAINED predictions
# ═══════════════════════════════════════════════════

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from difflib import SequenceMatcher
import seaborn as sns
import matplotlib.pyplot as plt

FastLanguageModel.for_inference(model)

# ─── The 10 valid departments ───
DEPARTMENTS = [
    'Billing and Payments', 'Customer Service', 'General Inquiry',
    'Human Resources', 'IT Support', 'Product Support',
    'Returns and Exchanges', 'Sales and Pre-Sales',
    'Service Outages and Maintenance', 'Technical Support'
]

dept_str = ", ".join(DEPARTMENTS)

def find_closest_department(pred):
    """Match prediction to closest valid department"""
    pred_lower = pred.lower().strip()

    # Exact match
    for dept in DEPARTMENTS:
        if dept.lower() == pred_lower:
            return dept

    # Contains match
    for dept in DEPARTMENTS:
        if dept.lower() in pred_lower or pred_lower in dept.lower():
            return dept

    # Fuzzy match (find most similar)
    best_match = None
    best_score = 0
    for dept in DEPARTMENTS:
        score = SequenceMatcher(None, pred_lower, dept.lower()).ratio()
        if score > best_score:
            best_score = score
            best_match = dept

    return best_match

# ─── Evaluate ───
test_df = pd.read_csv('/content/drive/MyDrive/ticket-classifier-data/test.csv')

y_true = []
y_pred = []

print(f"Evaluating on {len(test_df)} test tickets...\n")

for i, row in test_df.iterrows():
    messages = [
        {"role": "system", "content": f"You are an IT support ticket classifier.\nClassify into one of these departments ONLY: {dept_str}\nRespond with the department name only."},
        {"role": "user", "content": row['Body']}
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=20, do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()

    matched = find_closest_department(response)

    y_true.append(row['Department'])
    y_pred.append(matched)

    if (i + 1) % 500 == 0:
        current_acc = accuracy_score(y_true, y_pred)
        print(f"  {i+1}/{len(test_df)} done... current accuracy: {current_acc:.2%}")

# ─── Results ───
acc = accuracy_score(y_true, y_pred)

print("\n" + "=" * 60)
print(f"  LLM FINE-TUNING RESULTS (3000 training samples)")
print(f"  Test Accuracy: {acc:.4f}")
print("=" * 60)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, labels=DEPARTMENTS))

# ─── Confusion Matrix ───
cm = confusion_matrix(y_true, y_pred, labels=DEPARTMENTS)

plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=DEPARTMENTS, yticklabels=DEPARTMENTS)
plt.title(f'LLaMA 3.1 Fine-Tuned (3K samples)\nTest Accuracy: {acc:.4f}',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ─── Sample Predictions ───
print("\n── Sample Predictions ──\n")
for i in range(15):
    status = "✓" if y_true[i] == y_pred[i] else "✗"
    print(f"  {status} True: {y_true[i]:<35s} Pred: {y_pred[i]}")

**third** try

In [ ]:
# ═══════════════════════════════════════════════════
# Attempt 3: Using the approach that WORKED before
# + Balanced sampling (new)
# + Better system message (new)
# Quick test on 500 samples
# ═══════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import pandas as pd
import json
import os
import random
from sklearn.model_selection import train_test_split
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# ─── 1. Load and split ───
df = pd.read_csv('/content/drive/MyDrive/IT Support Ticket Data.csv', index_col=0)
df = df.dropna(subset=['Body'])
df = df[['Body', 'Department']]

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['Department'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['Department'], random_state=42
)

DEPARTMENTS = sorted(df['Department'].unique().tolist())

# ─── 2. BALANCED sampling: 50 per department = 500 total ───
samples_per_dept = 50
balanced_train = []
for dept in DEPARTMENTS:
    dept_samples = train_df[train_df['Department'] == dept].head(samples_per_dept)
    balanced_train.append(dept_samples)

train_balanced = pd.concat(balanced_train).sample(frac=1, random_state=42)
print(f"Balanced train: {len(train_balanced)} samples")
print(train_balanced['Department'].value_counts().to_string())

# ─── 3. Save as JSON (same format that worked before) ───
system_message = (
    "You are an IT support ticket classifier.\n"
    "You must classify the ticket into exactly one of these departments:\n"
    "- Billing and Payments\n"
    "- Customer Service\n"
    "- General Inquiry\n"
    "- Human Resources\n"
    "- IT Support\n"
    "- Product Support\n"
    "- Returns and Exchanges\n"
    "- Sales and Pre-Sales\n"
    "- Service Outages and Maintenance\n"
    "- Technical Support\n"
    "Respond with ONLY the department name, nothing else."
)

train_data = []
for _, row in train_balanced.iterrows():
    train_data.append({
        "system": system_message,
        "instruction": row['Body'],
        "input": "",
        "output": row['Department'],
        "history": []
    })

save_dir = '/content/drive/MyDrive/ticket-classifier-data'
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, 'train_balanced.json'), 'w') as f:
    json.dump(train_data, f, ensure_ascii=False)

test_df.to_csv(os.path.join(save_dir, 'test.csv'), index=False)

print(f"\nSaved {len(train_data)} samples to train_balanced.json")

# ─── 4. Load from JSON (same way that worked before) ───
train_dataset = load_dataset('json',
    data_files=os.path.join(save_dir, 'train_balanced.json'),
    split='train'
)

print(f"Loaded dataset: {len(train_dataset)} samples")
print(f"Columns: {train_dataset.column_names}")

# ─── 5. Formatting function (same style that worked before) ───
def formatting_func(example):
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{example['system']}<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    return [text]

# Show example
sample_text = formatting_func(train_dataset[0])
print(f"\n--- Example (first 400 chars) ---")
print(sample_text[0][:400])

# ─── 6. Train ───
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    formatting_func = formatting_func,
    max_seq_length = 1024,
    packing = False,
    args = TrainingArguments(
        output_dir = "/content/attempt3",
        num_train_epochs = 5,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        warmup_steps = 10,
        logging_steps = 10,
        save_strategy = "no",
        eval_strategy = "no",
        fp16 = True,
        optim = "adamw_8bit",
        seed = 42,
        report_to = "none",
    ),
)

print(f"\nTraining started...")
print(f"Epochs: 5, Batch: 2, Grad accum: 4")
print(f"This should take about 5-10 minutes\n")

trainer.train()
print("\nTraining complete!")

# ─── 7. Quick inference test ───
FastLanguageModel.for_inference(model)

test_tickets = [
    "My printer is not working and I need someone to fix it urgently.",
    "I want to return the product I bought last week.",
    "There is an error in my billing statement, I was charged twice.",
]

print("\n── Quick Inference Test ──\n")

for ticket in test_tickets:
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": ticket}
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=20, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

    print(f"  Ticket: {ticket[:60]}...")
    print(f"  Pred:   {response}\n")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from difflib import SequenceMatcher
import seaborn as sns
import matplotlib.pyplot as plt

FastLanguageModel.for_inference(model)

DEPARTMENTS = [
    'Billing and Payments', 'Customer Service', 'General Inquiry',
    'Human Resources', 'IT Support', 'Product Support',
    'Returns and Exchanges', 'Sales and Pre-Sales',
    'Service Outages and Maintenance', 'Technical Support'
]

system_msg = (
    "You are an IT support ticket classifier.\n"
    "Classify into exactly one department:\n"
    "Billing and Payments, Customer Service, General Inquiry, "
    "Human Resources, IT Support, Product Support, "
    "Returns and Exchanges, Sales and Pre-Sales, "
    "Service Outages and Maintenance, Technical Support.\n"
    "Respond with ONLY the department name."
)

def match_dept(pred):
    p = pred.lower().strip()
    for d in DEPARTMENTS:
        if d.lower() == p: return d
    for d in DEPARTMENTS:
        if d.lower() in p or p in d.lower(): return d
    return max(DEPARTMENTS, key=lambda d: SequenceMatcher(None, p, d.lower()).ratio())

# ─── 1000 test samples (100 per department) ───
test_df = pd.read_csv('/content/drive/MyDrive/ticket-classifier-data/test.csv')
test_1k = pd.concat([test_df[test_df['Department']==d].head(100) for d in DEPARTMENTS])

y_true, y_pred, raw = [], [], []
print(f"Evaluating {len(test_1k)} tickets...\n")

for i, (_, row) in enumerate(test_1k.iterrows()):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": row['Body'][:300]}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=15, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

    y_true.append(row['Department'])
    y_pred.append(match_dept(resp))
    raw.append(resp)

    if (i+1) % 200 == 0:
        print(f"  {i+1}/{len(test_1k)}... acc: {accuracy_score(y_true, y_pred):.2%}")

# ─── Results ───
acc = accuracy_score(y_true, y_pred)
print(f"\n{'='*60}")
print(f"  LLaMA 3.1 + LoRA (500 balanced train)")
print(f"  Test Accuracy on {len(test_1k)} samples: {acc:.4f}")
print(f"{'='*60}")
print(classification_report(y_true, y_pred, labels=DEPARTMENTS, zero_division=0))

# ─── Confusion Matrix ───
cm = confusion_matrix(y_true, y_pred, labels=DEPARTMENTS)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=DEPARTMENTS, yticklabels=DEPARTMENTS)
plt.title(f'LLaMA 3.1 + LoRA (500 balanced samples)\nTest Accuracy: {acc:.4f}',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ─── Samples ───
print("\n── Sample Predictions ──\n")
for i in range(20):
    s = "✓" if y_true[i]==y_pred[i] else "✗"
    print(f"  {s} True: {y_true[i]:<35s} Raw: {raw[i]:<30s} Match: {y_pred[i]}")

fourth try

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from difflib import SequenceMatcher
from trl import SFTTrainer
from transformers import TrainingArguments

# ═══════════════════════════════════════════════════
# 1. LOAD AND SPLIT
# ═══════════════════════════════════════════════════
df = pd.read_csv('/content/drive/MyDrive/IT Support Ticket Data.csv', index_col=0)
df = df.dropna(subset=['Body'])
df = df[['Body', 'Department']]

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['Department'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['Department'], random_state=42)

DEPARTMENTS = sorted(df['Department'].unique().tolist())

# ═══════════════════════════════════════════════════
# 2. BALANCED SAMPLING: 50 per department = 500
# ═══════════════════════════════════════════════════
balanced = pd.concat([
    train_df[train_df['Department'] == d].head(50) for d in DEPARTMENTS
]).sample(frac=1, random_state=42)

print(f"Balanced train: {len(balanced)} samples")

# ═══════════════════════════════════════════════════
# 3. BUILD TEXT COLUMN (NOT formatting_func)
# The key fix: formatting_func was causing "Num examples = 6"
# Instead we build the text ourselves and use dataset_text_field
# ═══════════════════════════════════════════════════
system_msg = (
    "You are an IT support ticket classifier.\n"
    "Classify into exactly one department:\n"
    "Billing and Payments, Customer Service, General Inquiry, "
    "Human Resources, IT Support, Product Support, "
    "Returns and Exchanges, Sales and Pre-Sales, "
    "Service Outages and Maintenance, Technical Support.\n"
    "Respond with ONLY the department name."
)

EOS = tokenizer.eos_token

texts = []
for _, row in balanced.iterrows():
    t = (
        "<|start_header_id|>system<|end_header_id|>\n\n"
        + system_msg
        + "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
        + row['Body'][:300]
        + "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        + row['Department'] + EOS
    )
    texts.append(t)

train_dataset = Dataset.from_dict({"text": texts})
print(f"Dataset: {len(train_dataset)} samples")

# ═══════════════════════════════════════════════════
# 4. TRAIN
# ═══════════════════════════════════════════════════
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    packing = False,
    dataset_num_proc = 2,
    args = TrainingArguments(
        output_dir = "/content/attempt4",
        num_train_epochs = 5,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",
        warmup_steps = 10,
        logging_steps = 20,
        save_strategy = "no",
        eval_strategy = "no",
        fp16 = True,
        optim = "adamw_8bit",
        seed = 42,
        report_to = "none",
    ),
)

total_steps = len(train_dataset) * 5 // (1 * 4)  # samples * epochs / (batch * grad_accum)
print(f"\nExpected training steps: ~{total_steps}")
print(f"If you see much fewer steps, the data is not loading correctly.")
print("Training started...\n")

trainer.train()
print("\nTraining complete!")

# ═══════════════════════════════════════════════════
# 5. QUICK TEST
# ═══════════════════════════════════════════════════
FastLanguageModel.for_inference(model)

tickets = [
    ("My printer is not working.", "IT Support"),
    ("I want to return my product.", "Returns and Exchanges"),
    ("Billing error on my statement.", "Billing and Payments"),
    ("Server outage affecting users.", "Service Outages and Maintenance"),
    ("Need info about product features.", "Product Support"),
]

print("\n── Quick Test ──\n")
correct = 0
for ticket, expected in tickets:
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": ticket}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=15, do_sample=False)
    pred = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    match = "✓" if expected.lower() in pred.lower() else "✗"
    if match == "✓": correct += 1
    print(f"  {match} Expected: {expected:<35s} Got: {pred}")

print(f"\nQuick test: {correct}/{len(tickets)} correct")

# ═══════════════════════════════════════════════════
# 6. EVALUATE ON 200 TEST SAMPLES
# ═══════════════════════════════════════════════════
def match_dept(pred):
    p = pred.lower().strip()
    for d in DEPARTMENTS:
        if d.lower() == p: return d
    for d in DEPARTMENTS:
        if d.lower() in p or p in d.lower(): return d
    return max(DEPARTMENTS, key=lambda d: SequenceMatcher(None, p, d.lower()).ratio())

test_mini = pd.concat([test_df[test_df['Department']==d].head(20) for d in DEPARTMENTS])

y_true, y_pred, raw = [], [], []
print(f"\nEvaluating {len(test_mini)} test tickets...\n")

for i, (_, row) in enumerate(test_mini.iterrows()):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": row['Body'][:300]}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=15, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

    y_true.append(row['Department'])
    y_pred.append(match_dept(resp))
    raw.append(resp)

    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(test_mini)}... acc: {accuracy_score(y_true, y_pred):.2%}")

acc = accuracy_score(y_true, y_pred)
print(f"\n{'='*60}")
print(f"  ATTEMPT 4 RESULTS (500 balanced → {len(test_mini)} test)")
print(f"  Test Accuracy: {acc:.4f}")
print(f"{'='*60}")
print(classification_report(y_true, y_pred, labels=DEPARTMENTS, zero_division=0))

print("── Samples ──\n")
for i in range(15):
    s = "✓" if y_true[i]==y_pred[i] else "✗"
    print(f"  {s} True: {y_true[i]:<35s} Raw: {raw[i]:<25s} Match: {y_pred[i]}")

test_df.to_csv('/content/drive/MyDrive/ticket-classifier-data/test.csv', index=False)

In [ ]:
# ═══════════════════════════════════════════════════
# Evaluation on 500 test samples + Confusion Matrix
# ═══════════════════════════════════════════════════

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from difflib import SequenceMatcher
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

FastLanguageModel.for_inference(model)

DEPARTMENTS = [
    'Billing and Payments', 'Customer Service', 'General Inquiry',
    'Human Resources', 'IT Support', 'Product Support',
    'Returns and Exchanges', 'Sales and Pre-Sales',
    'Service Outages and Maintenance', 'Technical Support'
]

system_msg = (
    "You are an IT support ticket classifier.\n"
    "Classify into exactly one department:\n"
    "Billing and Payments, Customer Service, General Inquiry, "
    "Human Resources, IT Support, Product Support, "
    "Returns and Exchanges, Sales and Pre-Sales, "
    "Service Outages and Maintenance, Technical Support.\n"
    "Respond with ONLY the department name."
)

def match_dept(pred):
    p = pred.lower().strip()
    for d in DEPARTMENTS:
        if d.lower() == p: return d
    for d in DEPARTMENTS:
        if d.lower() in p or p in d.lower(): return d
    return max(DEPARTMENTS, key=lambda d: SequenceMatcher(None, p, d.lower()).ratio())

# ─── 500 balanced test samples (50 per department) ───
test_df = pd.read_csv('/content/drive/MyDrive/ticket-classifier-data/test.csv')

test_500 = pd.concat([
    test_df[test_df['Department'] == d].head(50) for d in DEPARTMENTS
])

print(f"Evaluating {len(test_500)} test tickets (50 per department)...\n")

y_true, y_pred, raw = [], [], []

for i, (_, row) in enumerate(test_500.iterrows()):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": row['Body'][:300]}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=15, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

    y_true.append(row['Department'])
    y_pred.append(match_dept(resp))
    raw.append(resp)

    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(test_500)}... acc: {accuracy_score(y_true, y_pred):.2%}")

# ─── Results ───
acc = accuracy_score(y_true, y_pred)

print(f"\n{'='*60}")
print(f"  LLaMA 3.1 + LoRA (500 balanced train)")
print(f"  Test Accuracy: {acc:.4f}")
print(f"{'='*60}")
print(f"\nClassification Report:\n")
print(classification_report(y_true, y_pred, labels=DEPARTMENTS, zero_division=0))

# ─── Confusion Matrix ───
cm = confusion_matrix(y_true, y_pred, labels=DEPARTMENTS)

plt.figure(figsize=(14, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=DEPARTMENTS, yticklabels=DEPARTMENTS
)
plt.title(f'LLaMA 3.1 + LoRA Fine-Tuned (500 balanced samples)\nTest Accuracy: {acc:.4f}',
          fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('llm_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# ─── Sample Predictions ───
print("\n── Sample Predictions (first 20) ──\n")
for i in range(min(20, len(y_true))):
    s = "✓" if y_true[i] == y_pred[i] else "✗"
    print(f"  {s} True: {y_true[i]:<35s} Raw: {raw[i]:<30s} Match: {y_pred[i]}")

لما تيجي تكتب نبدأ ب bow navies bayes
بعدين نبدأ ندخل ف بقية الموديلات
تشرح ال word 2 vec مع rndom forest

بعدين نشرح ال llama و ليه اخدتها
واشرح تأثيرها ب domian addaptopn وبدون

fivth try

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from difflib import SequenceMatcher
from trl import SFTTrainer
from transformers import TrainingArguments

# =========================================================
# 1) LOAD
# =========================================================
df = pd.read_csv('/content/drive/MyDrive/IT Support Ticket Data.csv', index_col=0)
df = df.dropna(subset=['Body']).copy()
df = df[['Body', 'Department']]

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['Department'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['Department'], random_state=42
)

DEPARTMENTS = sorted(df['Department'].unique().tolist())

# short labels بدل الأسماء الطويلة
label_map = {
    "Billing and Payments": "BILLING",
    "Customer Service": "CUSTOMER",
    "General Inquiry": "GENERAL",
    "Human Resources": "HR",
    "IT Support": "IT",
    "Product Support": "PRODUCT",
    "Returns and Exchanges": "RETURNS",
    "Sales and Pre-Sales": "SALES",
    "Service Outages and Maintenance": "OUTAGE",
    "Technical Support": "TECH",
}
inv_label_map = {v: k for k, v in label_map.items()}

# =========================================================
# 2) BALANCED RANDOM SAMPLE
# استخدم 150-300 لكل قسم لو الذاكرة تسمح
# =========================================================
per_class = 100
train_balanced = pd.concat([
    g.sample(n=min(per_class, len(g)), random_state=42)
    for _, g in train_df.groupby("Department")
]).sample(frac=1, random_state=42).reset_index(drop=True)

val_balanced = pd.concat([
    g.sample(n=min(40, len(g)), random_state=42)
    for _, g in val_df.groupby("Department")
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Train balanced:", len(train_balanced))
print("Val balanced:", len(val_balanced))

# =========================================================
# 3) PROMPT FORMAT
# نفس الصياغة في التدريب والاختبار
# =========================================================
system_msg = (
    "You are a classifier for IT support tickets.\n"
    "Return exactly one label from this list only:\n"
    "BILLING, CUSTOMER, GENERAL, HR, IT, PRODUCT, RETURNS, SALES, OUTAGE, TECH.\n"
    "Do not explain."
)

def build_example(body, label_code=None):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": body[:700]}
    ]
    if label_code is not None:
        messages.append({"role": "assistant", "content": label_code})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return text

train_texts = [
    build_example(row.Body, label_map[row.Department])
    for row in train_balanced.itertuples()
]
val_texts = [
    build_example(row.Body, label_map[row.Department])
    for row in val_balanced.itertuples()
]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))

# =========================================================
# 4) TRAIN
# =========================================================
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=768,
    packing=False,
    dataset_num_proc=2,
   args=TrainingArguments(
    output_dir="/content/llama_ticket_cls",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
)
,
)

# لو نسختك من unsloth تدعم train_on_responses_only استخدمها
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
        response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
    )
    print("Using response-only loss.")
except Exception as e:
    print("Response-only loss not enabled:", e)

trainer.train()
print("Training complete!")

# =========================================================
# 5) INFERENCE
# =========================================================
FastLanguageModel.for_inference(model)

valid_codes = list(inv_label_map.keys())

def normalize_pred(pred):
    p = pred.strip().upper().split()[0].replace(".", "").replace(",", "")
    if p in valid_codes:
        return inv_label_map[p]

    best = max(valid_codes, key=lambda x: SequenceMatcher(None, p, x).ratio())
    return inv_label_map[best]

def predict_ticket(body):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": body[:700]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=3,
        do_sample=False,
        temperature=0.0,
    )
    pred = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return pred, normalize_pred(pred)

# =========================================================
# 6) QUICK TEST
# =========================================================
tickets = [
    ("My printer is not working.", "IT Support"),
    ("I want to return my product.", "Returns and Exchanges"),
    ("Billing error on my statement.", "Billing and Payments"),
    ("Server outage affecting users.", "Service Outages and Maintenance"),
    ("Need info about product features.", "Product Support"),
]

correct = 0
for text, expected in tickets:
    raw, pred = predict_ticket(text)
    ok = pred == expected
    correct += int(ok)
    print(("✓" if ok else "✗"), "Expected:", expected, "| Raw:", raw, "| Pred:", pred)

print(f"\nQuick test: {correct}/{len(tickets)}")

# =========================================================
# 7) EVAL
# =========================================================
test_mini = pd.concat([
    g.sample(n=min(30, len(g)), random_state=42)
    for _, g in test_df.groupby("Department")
]).reset_index(drop=True)

y_true, y_pred, raw_preds = [], [], []

for i, row in enumerate(test_mini.itertuples(), start=1):
    raw, pred = predict_ticket(row.Body)
    y_true.append(row.Department)
    y_pred.append(pred)
    raw_preds.append(raw)

    if i % 50 == 0:
        print(f"{i}/{len(test_mini)} - acc: {accuracy_score(y_true, y_pred):.2%}")

acc = accuracy_score(y_true, y_pred)
print(f"\nTest Accuracy: {acc:.4f}")
print(classification_report(y_true, y_pred, labels=DEPARTMENTS, zero_division=0))


الدكتوره كانت قالتلي اعمل hyperparmeter tunning ف دا برضو
